# 06 Structural Estimation

This notebook estimates the structural search/preference model after the first five rebuild notebooks have prepared interaction features, beliefs, GNN/MLP head predictions, and head weights.

The goal is to estimate three connected objects:

| Object | Meaning | Main output |
|---|---|---|
| `theta` | User-category utility means and user search costs | `mu_{ik}`, `c_i` |
| `phi` | Watch-ratio measurement model for latent states `z=0,1` | State-specific means and variances for log watch ratio |
| `r_itj` | E-step responsibility, posterior probability of state 1 | `Pr(z_itj=1 | data, theta, phi)` |

The notebook includes the bug fixes listed in `structural_estimation_bugs_and_scale_normalization.md`:

1. The measurement variance is always `exp(eta) + min_var`, evaluated through `logaddexp`.
2. The log-variance M-step has no hard clipping and uses a gradient consistent with the objective.
3. The best-likelihood snapshot can be the post-`phi`, pre-`theta` state.
4. The threshold solver validates and expands its bisection bracket.
5. Search costs are floored by `COST_FLOOR` and are never allowed to be exactly zero.
6. The approximate fixed-threshold `theta` update is guarded as a generalized EM step.
7. Every user's UNKNOWN category is normalized to a `N(0,1)` utility baseline by shifting and scaling all scores, sigmas, and initial costs for that user by the same factor.


## Roadmap

1. Load the processed KuaiRec interaction table and the vanilla Dirichlet belief table.
2. Build an estimation sample after burst burn-in and minimum-user-observation filtering.
3. Convert four behavioral heads into a utility score proxy using the PL head weights from notebook 04b when available.
4. Normalize each user's score scale so the UNKNOWN category is the location/scale reference.
5. Build session-level beliefs, compact interaction arrays, and validation checks.
6. Define the threshold solver, measurement model, E-step, M-step, and EM driver.
7. Run an optional smoke test, then the full-data structural estimation.
8. Save diagnostics needed to audit the run later.


## 1. Estimation Sample

The structural model is estimated on interactions after a configurable burst burn-in. The burn-in removes the first calendar days inside each burst so that history and belief variables are less dominated by initialization artifacts.

The observed measurement is

$$
s_{itj}=\log\left(\max\left(\mathrm{watch\_ratio}_{itj},\epsilon\right)\right).
$$

Here `WATCH_RATIO_FLOOR` is the small positive floor $\epsilon$.


In [ ]:

from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
BASE = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed"
NEW_CODE_DIR = PROJECT_ROOT / "python_code_new"
OUTPUT_DIR = NEW_CODE_DIR / "outputs"
DOCS_DIR = NEW_CODE_DIR / "docs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = BASE / "processed_data.parquet"
BELIEF_PATH = BASE / "valid_user_belief.parquet"
HEAD_WEIGHT_PATH = OUTPUT_DIR / "head_weight_pl_weights.json"

# Estimation sample and measurement defaults.
BURN_IN_DAYS = 2
MIN_OBS_PER_USER = 2000
WATCH_RATIO_FLOOR = 1e-3

# Utility score proxy used by the structural model.
# Only this weight specification changes the score proxy; the structural
# estimation procedure below is unchanged.
HEAD_WEIGHT_SPEC = "naive"  # choices: "naive", "constrained", "unconstrained"
NAIVE_SCORE_WEIGHTS = {
    "w_complete": 1.0,
    "w_long": 1.0,
    "w_rewatch": 1.0,
    "w_neg": -1.0,
}


def load_head_weight_payload(path, spec):
    """Load constrained/unconstrained head weights from the PL-weight JSON file."""
    spec = str(spec)
    if spec not in {"constrained", "unconstrained"}:
        raise ValueError("HEAD_WEIGHT_SPEC must be 'naive', 'constrained', or 'unconstrained'")

    with open(path, "r") as f:
        payload = json.load(f)

    weights = None

    # For the PL file, the constrained fit was saved as a diagnostic object.
    if spec == "constrained" and "constrained_diagnostic_weights" in payload:
        weights = payload["constrained_diagnostic_weights"]

    # Nested diagnostic schema.
    if weights is None and (
        "optimization_diagnostics" in payload
        and isinstance(payload["optimization_diagnostics"], dict)
        and spec in payload["optimization_diagnostics"]
        and isinstance(payload["optimization_diagnostics"][spec], dict)
        and "weights" in payload["optimization_diagnostics"][spec]
    ):
        weights = payload["optimization_diagnostics"][spec]["weights"]

    # Direct convenience schema.
    direct_key = f"{spec}_weights"
    if weights is None and direct_key in payload:
        weights = payload[direct_key]

    # Top-level weights represent the primary fit, usually unconstrained for the PL file.
    if weights is None and "weights" in payload:
        primary_fit = payload.get("primary_fit")
        if primary_fit is None and isinstance(payload.get("objective"), dict):
            primary_fit = payload["objective"].get("primary_fit")
        if primary_fit is None or str(primary_fit) == spec:
            weights = payload["weights"]

    if weights is None:
        raise KeyError(f"Could not find {spec!r} weights in {path}; available keys: {list(payload.keys())}")

    return {str(k): float(v) for k, v in weights.items()}, payload


def normalize_head_weight_keys(raw_weights):
    """Normalize weight keys to canonical head names: complete/long/rewatch/neg."""
    key_map = {
        "w_complete": "complete",
        "w_long": "long",
        "w_rewatch": "rewatch",
        "w_neg": "neg",
        "p_complete": "complete",
        "p_long": "long",
        "p_rewatch": "rewatch",
        "p_neg": "neg",
        "y_complete": "complete",
        "y_long": "long",
        "y_rewatch": "rewatch",
        "y_neg": "neg",
        "complete": "complete",
        "long": "long",
        "rewatch": "rewatch",
        "neg": "neg",
    }

    out = {}
    for key, value in raw_weights.items():
        if key not in key_map:
            raise KeyError(f"Unknown head-weight key: {key}")
        out[key_map[key]] = float(value)

    required = {"complete", "long", "rewatch", "neg"}
    missing = required - set(out)
    if missing:
        raise KeyError(f"Missing required head weights: {missing}")
    return out


CANONICAL_HEAD_TO_DATA_COLUMN = {
    "complete": "y_complete",
    "long": "y_long",
    "rewatch": "y_rewatch",
    "neg": "y_neg",
}


if HEAD_WEIGHT_SPEC == "naive":
    raw_score_weights = dict(NAIVE_SCORE_WEIGHTS)
    head_weight_payload = {"source": "manual naive weights"}
    SCORE_WEIGHT_SOURCE = "manual::naive_(1,1,1,-1)"
elif HEAD_WEIGHT_SPEC in {"constrained", "unconstrained"}:
    raw_score_weights, head_weight_payload = load_head_weight_payload(HEAD_WEIGHT_PATH, HEAD_WEIGHT_SPEC)
    SCORE_WEIGHT_SOURCE = f"{HEAD_WEIGHT_PATH}::{HEAD_WEIGHT_SPEC}"
else:
    raise ValueError("HEAD_WEIGHT_SPEC must be 'naive', 'constrained', or 'unconstrained'")

canonical_score_weights = normalize_head_weight_keys(raw_score_weights)
SCORE_WEIGHTS = {CANONICAL_HEAD_TO_DATA_COLUMN[key]: value for key, value in canonical_score_weights.items()}

SCORE_SCALE = 10.0
SCORE_REFERENCE_CATEGORY_ID = -124  # UNKNOWN category; used as utility location/scale normalization.

# Score-scale and variance controls.
VAR_SHRINK_N0 = 10.0
RELIABLE_VAR_MIN_N = 10
VAR_FLOOR = 1e-6
SCALE_FLOOR = 1e-6

# Search costs are never allowed to be zero.
COST_FLOOR = 1e-6
INITIAL_SEARCH_COST_RAW = 0.5
GEM_TOL = 1e-6
THRESHOLD_RESIDUAL_TOL = 1e-5

print("head weight spec:", HEAD_WEIGHT_SPEC)
print("score weight source:", SCORE_WEIGHT_SOURCE)
print(json.dumps(SCORE_WEIGHTS, indent=2))


In [ ]:

# Smoke tests for the PL head-weight JSON schema used by this notebook.
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    pl_path = tmpdir / "head_weight_pl_weights.json"
    pl_path.write_text(json.dumps({
        "weights": {"w_complete": 2, "w_long": 0.5, "w_rewatch": -0.5, "w_neg": -2},
        "objective": {"primary_fit": "unconstrained"},
        "optimization_diagnostics": {
            "constrained": {"weights": {"w_complete": 1, "w_long": 0, "w_rewatch": 0, "w_neg": -1}},
            "unconstrained": {"weights": {"w_complete": 2, "w_long": 0.5, "w_rewatch": -0.5, "w_neg": -2}},
        },
        "constrained_diagnostic_weights": {"w_complete": 1, "w_long": 0, "w_rewatch": 0, "w_neg": -1},
    }))

    constrained_weights, _ = load_head_weight_payload(pl_path, spec="constrained")
    unconstrained_weights, _ = load_head_weight_payload(pl_path, spec="unconstrained")

    assert normalize_head_weight_keys(NAIVE_SCORE_WEIGHTS) == {"complete": 1.0, "long": 1.0, "rewatch": 1.0, "neg": -1.0}
    assert normalize_head_weight_keys(constrained_weights) == {"complete": 1.0, "long": 0.0, "rewatch": 0.0, "neg": -1.0}
    assert normalize_head_weight_keys(unconstrained_weights) == {"complete": 2.0, "long": 0.5, "rewatch": -0.5, "neg": -2.0}

print("head-weight JSON smoke tests passed")


In [ ]:
df_raw = pd.read_parquet(DATA_PATH)
# row_id is the original processed-data row index. Notebook 02 uses this same
# row order for GNN edge rows, so keeping row_id makes downstream target
# alignment exact.
df_raw.insert(0, "row_id", np.arange(len(df_raw), dtype=np.int64))
belief_raw = pd.read_parquet(BELIEF_PATH)

print("raw interactions:", df_raw.shape)
print("raw beliefs:", belief_raw.shape)
print("interaction columns:", df_raw.columns.tolist()[:25], "...")
print("belief columns:", belief_raw.columns.tolist())


### Burst Burn-In

For each `burst_id`, let `d_b^{min}` be the first date in that burst. With `BURN_IN_DAYS = B`, the notebook drops rows with dates in

$$
    \{d_b^{min}, d_b^{min}+1, \ldots, d_b^{min}+B-1\}.
$$

If `BURN_IN_DAYS = 0`, this step is skipped.


In [ ]:
def add_normalized_date(df, date_col="date"):
    out = df.copy()
    out["date_dt"] = pd.to_datetime(out[date_col].astype(str), format="%Y%m%d", errors="coerce").dt.normalize()
    return out


def drop_burst_burn_in(df, burn_in_days):
    """Drop the first `burn_in_days` calendar days within each burst."""
    if burn_in_days <= 0:
        return df.copy(), pd.Series(False, index=df.index)

    out = add_normalized_date(df)
    burst_start = out.groupby("burst_id", observed=True)["date_dt"].transform("min")
    burn_in_end = burst_start + pd.to_timedelta(burn_in_days - 1, unit="D")
    mask_drop = out["date_dt"].between(burst_start, burn_in_end)
    return out.loc[~mask_drop].copy(), mask_drop


df_burn, burn_mask = drop_burst_burn_in(df_raw, BURN_IN_DAYS)

print(f"burn-in days: {BURN_IN_DAYS}")
print(f"rows: {len(df_raw):,} -> {len(df_burn):,} (dropped {int(burn_mask.sum()):,})")
print("dropped share:", float(burn_mask.mean()))

daily_counts = (
    add_normalized_date(df_raw)
    .assign(dropped_by_burn_in=burn_mask.to_numpy())
    .groupby(["burst_id", "date_dt"], observed=True)
    .agg(n=("user_id", "size"), dropped_by_burn_in=("dropped_by_burn_in", "any"))
    .reset_index()
)
daily_counts


In [ ]:
user_counts = df_burn["user_id"].value_counts()
valid_users = user_counts[user_counts >= MIN_OBS_PER_USER].index.astype(int)

df_valid = df_burn[df_burn["user_id"].isin(valid_users)].copy()

print("min observations per user:", MIN_OBS_PER_USER)
print("valid users:", len(valid_users))
print("valid interactions:", df_valid.shape)
print("interactions per valid user:")
print(df_valid["user_id"].value_counts().describe())


In [ ]:
df_valid["watch_ratio"] = pd.to_numeric(df_valid["watch_ratio"], errors="coerce")
df_valid["log_watch_ratio"] = np.log(
    np.maximum(df_valid["watch_ratio"].to_numpy(dtype=float), WATCH_RATIO_FLOOR)
)

print(df_valid[["watch_ratio", "log_watch_ratio"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


### Score Proxy and Per-User UNKNOWN Normalization

The structural utility is not directly observed, so the notebook begins from a score proxy built from the four response heads:

$$
    q_{itj}=S\left(
        w_{complete} y^{complete}_{itj}
        + w_{long} y^{long}_{itj}
        + w_{rewatch} y^{rewatch}_{itj}
        + w_{neg} y^{neg}_{itj}
    \right).
$$

`SCORE_SCALE = S`. The active weight vector is controlled by `HEAD_WEIGHT_SPEC` in the setup cell. `HEAD_WEIGHT_SPEC = "naive"` uses `(1, 1, 1, -1)`; changing it to `"constrained"` or `"unconstrained"` loads the corresponding PL weights from `head_weight_pl_weights.json`.

The model uses the UNKNOWN category as the user-specific utility origin and scale. For user `i`, let `u` denote the UNKNOWN category. Define a raw user shift

$$
    a_i = E[q_{itj}\mid i, k=u],
$$

with a global fallback for users without UNKNOWN observations. Define a raw user scale

$$
    s_i =
    \begin{cases}
    \widehat\sigma_{iu}, & \text{if UNKNOWN has enough observations and positive variance},\\
    \operatorname{median}_{k\in\mathcal R_i}\widehat\sigma_{ik}, & \text{if user } i \text{ has another reliable category},\\
    \operatorname{median}_{(i,k)\in\mathcal R}\widehat\sigma_{ik}, & \text{otherwise},
    \end{cases}
$$

where `R_i` is the set of reliable user-category cells. The normalized score is

$$
    \widetilde q_{itj}=\frac{q_{itj}-a_i}{s_i}.
$$

This means the UNKNOWN category is anchored at mean zero and, when reliable, variance one. The same scale `s_i` is also used later for `sigma_{ik}` and the initial search cost `c_i`, so every utility object for the user is in the same normalized units.


In [ ]:
missing_heads = [name for name in SCORE_WEIGHTS if name not in df_valid.columns]
if missing_heads:
    raise KeyError(f"Missing score-head columns: {missing_heads}")

score_raw = np.zeros(len(df_valid), dtype=np.float64)
for name, weight in SCORE_WEIGHTS.items():
    score_raw += float(weight) * pd.to_numeric(df_valid[name], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
df_valid["score_raw"] = float(SCORE_SCALE) * score_raw

# Raw user-category score moments are used only to compute the per-user normalization scale.
raw_user_cat_moments = (
    df_valid
    .groupby(["user_id", "i_cat_level1_id"], observed=True)["score_raw"]
    .agg(
        raw_mean_score="mean",
        raw_var_score=lambda x: x.var(ddof=0),
        raw_n_score="size",
    )
    .reset_index()
)

raw_mean_wide = (
    raw_user_cat_moments
    .pivot(index="user_id", columns="i_cat_level1_id", values="raw_mean_score")
)
raw_var_wide = (
    raw_user_cat_moments
    .pivot(index="user_id", columns="i_cat_level1_id", values="raw_var_score")
)
raw_n_wide = (
    raw_user_cat_moments
    .pivot(index="user_id", columns="i_cat_level1_id", values="raw_n_score")
    .fillna(0.0)
)

all_users_in_score = np.sort(df_valid["user_id"].unique().astype(int))
raw_mean_wide = raw_mean_wide.reindex(index=all_users_in_score)
raw_var_wide = raw_var_wide.reindex(index=all_users_in_score)
raw_n_wide = raw_n_wide.reindex(index=all_users_in_score)

if SCORE_REFERENCE_CATEGORY_ID in raw_mean_wide.columns:
    unknown_mean_by_user = raw_mean_wide[SCORE_REFERENCE_CATEGORY_ID]
else:
    unknown_mean_by_user = pd.Series(np.nan, index=all_users_in_score, dtype=float)

global_unknown_mean = float(unknown_mean_by_user.dropna().mean()) if unknown_mean_by_user.notna().any() else float(df_valid["score_raw"].mean())
if not np.isfinite(global_unknown_mean):
    global_unknown_mean = 0.0

raw_var = raw_var_wide.to_numpy(dtype=np.float64)
raw_n = raw_n_wide.to_numpy(dtype=np.float64)
reliable_raw_var = (raw_n >= RELIABLE_VAR_MIN_N) & np.isfinite(raw_var) & (raw_var > VAR_FLOOR)

if reliable_raw_var.any():
    global_reliable_raw_sigma = float(np.sqrt(np.nanmedian(raw_var[reliable_raw_var])))
else:
    global_reliable_raw_sigma = float(np.nanstd(df_valid["score_raw"].to_numpy(dtype=np.float64)))
if not np.isfinite(global_reliable_raw_sigma) or global_reliable_raw_sigma <= SCALE_FLOOR:
    global_reliable_raw_sigma = 1.0

user_median_raw_sigma = np.full(len(all_users_in_score), global_reliable_raw_sigma, dtype=np.float64)
for i in range(len(all_users_in_score)):
    vals = raw_var[i, reliable_raw_var[i]]
    if vals.size > 0:
        user_median_raw_sigma[i] = float(np.sqrt(np.nanmedian(vals)))

unknown_scale_by_user = pd.Series(np.nan, index=all_users_in_score, dtype=float)
unknown_scale_reliable = pd.Series(False, index=all_users_in_score, dtype=bool)
if SCORE_REFERENCE_CATEGORY_ID in raw_var_wide.columns:
    unknown_var = raw_var_wide[SCORE_REFERENCE_CATEGORY_ID].to_numpy(dtype=np.float64)
    unknown_n = raw_n_wide[SCORE_REFERENCE_CATEGORY_ID].to_numpy(dtype=np.float64)
    unknown_ok = (unknown_n >= RELIABLE_VAR_MIN_N) & np.isfinite(unknown_var) & (unknown_var > VAR_FLOOR)
    unknown_scale_by_user.iloc[:] = np.where(unknown_ok, np.sqrt(unknown_var), np.nan)
    unknown_scale_reliable.iloc[:] = unknown_ok

utility_scale = unknown_scale_by_user.to_numpy(dtype=np.float64)
scale_source = np.full(len(all_users_in_score), "unknown_reliable", dtype=object)
missing_unknown_scale = ~np.isfinite(utility_scale) | (utility_scale <= SCALE_FLOOR)
utility_scale[missing_unknown_scale] = user_median_raw_sigma[missing_unknown_scale]
scale_source[missing_unknown_scale] = "user_reliable_median"
still_bad = ~np.isfinite(utility_scale) | (utility_scale <= SCALE_FLOOR)
utility_scale[still_bad] = global_reliable_raw_sigma
scale_source[still_bad] = "global_reliable_median"
utility_scale = np.maximum(utility_scale, SCALE_FLOOR)

utility_shift = unknown_mean_by_user.reindex(all_users_in_score).fillna(global_unknown_mean).to_numpy(dtype=np.float64)
shift_source = np.where(unknown_mean_by_user.reindex(all_users_in_score).notna().to_numpy(), "user_unknown_mean", "global_unknown_mean")

utility_norm_summary = pd.DataFrame({
    "user_id": all_users_in_score.astype(int),
    "utility_shift_raw_unknown_mean": utility_shift,
    "utility_scale_raw_sigma": utility_scale,
    "shift_source": shift_source,
    "scale_source": scale_source,
    "unknown_scale_reliable": unknown_scale_reliable.reindex(all_users_in_score).fillna(False).to_numpy(dtype=bool),
})

shift_map = dict(zip(utility_norm_summary["user_id"], utility_norm_summary["utility_shift_raw_unknown_mean"]))
scale_map = dict(zip(utility_norm_summary["user_id"], utility_norm_summary["utility_scale_raw_sigma"]))
df_valid["utility_shift_raw_unknown_mean"] = df_valid["user_id"].map(shift_map).astype(float)
df_valid["utility_scale_raw_sigma"] = df_valid["user_id"].map(scale_map).astype(float)
df_valid["score"] = (df_valid["score_raw"] - df_valid["utility_shift_raw_unknown_mean"]) / df_valid["utility_scale_raw_sigma"]

user_cat_score_moments = (
    df_valid
    .groupby(["user_id", "i_cat_level1_id"], observed=True)["score"]
    .agg(mean_score="mean", var_score=lambda x: x.var(ddof=0), n_score="size")
    .reset_index()
)

initial_cost_by_user = (float(INITIAL_SEARCH_COST_RAW) / utility_norm_summary["utility_scale_raw_sigma"].to_numpy(dtype=np.float64)).astype(np.float64)
initial_cost_by_user = np.maximum(initial_cost_by_user, COST_FLOOR)

print("score reference category:", SCORE_REFERENCE_CATEGORY_ID)
print("score weight source:", SCORE_WEIGHT_SOURCE)
print("users with UNKNOWN mean:", int((utility_norm_summary["shift_source"] == "user_unknown_mean").sum()))
print("users using global UNKNOWN mean fallback:", int((utility_norm_summary["shift_source"] == "global_unknown_mean").sum()))
print("scale source counts:")
print(utility_norm_summary["scale_source"].value_counts())
print("normalized score summary:")
print(df_valid["score"].describe())
print("UNKNOWN normalized mean/variance check for reliable UNKNOWN cells:")
unknown_norm = df_valid.loc[df_valid["i_cat_level1_id"].eq(SCORE_REFERENCE_CATEGORY_ID)]
if len(unknown_norm):
    display(
        unknown_norm.groupby("user_id", observed=True)["score"]
        .agg(mean="mean", var=lambda x: x.var(ddof=0), n="size")
        .describe()
    )
print("user-category score moments:", user_cat_score_moments.shape)
display(utility_norm_summary.head())
user_cat_score_moments.head()


### Shrink User-Category Utility Variances

For each normalized user-category score cell, the notebook estimates the utility standard deviation `sigma_ik`. Sparse cells are shrunk toward a user-level fallback variance:

$$
    \widehat v^{shrunk}_{ik}
    = \lambda_{ik}\widehat v_{ik} + (1-\lambda_{ik})v^{fallback}_i,
    \qquad
    \lambda_{ik}=\frac{n_{ik}}{n_{ik}+n_0}.
$$

Reliable cells keep their empirical variance. Sparse or missing cells borrow the user's median reliable variance, with a global median fallback. Because score normalization has already divided by `s_i`, a reliable UNKNOWN cell has `sigma_iu` approximately equal to one.


In [ ]:
user_ids = np.sort(df_valid["user_id"].unique().astype(int))
cat_ids = np.sort(df_valid["i_cat_level1_id"].unique().astype(int))

uid_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
cat_to_idx = {cid: idx for idx, cid in enumerate(cat_ids)}

N_users = len(user_ids)
K_cat = len(cat_ids)

print("N_users:", N_users)
print("K_cat:", K_cat)

UTILITY_REFERENCE_CATEGORY_ID = SCORE_REFERENCE_CATEGORY_ID
if UTILITY_REFERENCE_CATEGORY_ID not in cat_to_idx:
    raise ValueError(f"Utility reference category {UTILITY_REFERENCE_CATEGORY_ID} is not in cat_ids")
UTILITY_REFERENCE_CAT_IDX = int(cat_to_idx[UTILITY_REFERENCE_CATEGORY_ID])
print("utility reference category id:", UTILITY_REFERENCE_CATEGORY_ID)
print("utility reference category idx:", UTILITY_REFERENCE_CAT_IDX)

# Align per-user normalization summary and initial costs with the final user index.
utility_norm_summary = utility_norm_summary.set_index("user_id").reindex(user_ids).reset_index()
initial_cost_by_user = np.maximum(
    float(INITIAL_SEARCH_COST_RAW) / utility_norm_summary["utility_scale_raw_sigma"].to_numpy(dtype=np.float64),
    COST_FLOOR,
)

var_wide = (
    user_cat_score_moments
    .pivot(index="user_id", columns="i_cat_level1_id", values="var_score")
    .reindex(index=user_ids, columns=cat_ids)
)
n_wide = (
    user_cat_score_moments
    .pivot(index="user_id", columns="i_cat_level1_id", values="n_score")
    .reindex(index=user_ids, columns=cat_ids)
    .fillna(0.0)
)

var_emp = var_wide.to_numpy(dtype=float)
n_obs = n_wide.to_numpy(dtype=float)

reliable_mask = (n_obs >= RELIABLE_VAR_MIN_N) & np.isfinite(var_emp) & (var_emp > VAR_FLOOR)
shrink_mask = ~reliable_mask
global_reliable_median = float(np.nanmedian(var_emp[reliable_mask])) if reliable_mask.any() else 1.0
if not np.isfinite(global_reliable_median) or global_reliable_median <= VAR_FLOOR:
    global_reliable_median = 1.0

user_median_var = np.full(N_users, global_reliable_median, dtype=float)
for i in range(N_users):
    reliable_vals = var_emp[i, reliable_mask[i]]
    if reliable_vals.size > 0:
        user_median_var[i] = float(np.nanmedian(reliable_vals))

fallback_var = user_median_var[:, None]
var_emp_filled = np.where(np.isfinite(var_emp) & (var_emp > VAR_FLOOR), var_emp, fallback_var)

lambda_ik = np.where(shrink_mask, n_obs / (n_obs + VAR_SHRINK_N0), 1.0)
var_shrunk = lambda_ik * var_emp_filled + (1.0 - lambda_ik) * fallback_var
var_shrunk = np.maximum(var_shrunk, VAR_FLOOR)

sigmas_by_user = np.sqrt(var_shrunk).astype(np.float64)

unknown_sigma = sigmas_by_user[:, UTILITY_REFERENCE_CAT_IDX]
utility_norm_summary["initial_cost_normalized"] = initial_cost_by_user
utility_norm_summary["sigma_unknown_after_shrink"] = unknown_sigma

print("global reliable variance median after normalization:", global_reliable_median)
print("users with no reliable category:", int((~reliable_mask.any(axis=1)).sum()))
print("user-category cells shrunk:", int(shrink_mask.sum()), "of", int(shrink_mask.size))
print("sigma range:", float(sigmas_by_user.min()), float(sigmas_by_user.max()))
print("sigma median:", float(np.median(sigmas_by_user)))
print("UNKNOWN sigma summary after shrink:")
print(pd.Series(unknown_sigma).describe())

sigma_summary = pd.DataFrame({
    "n_obs": n_obs.ravel(),
    "var_emp": var_emp.ravel(),
    "var_shrunk": var_shrunk.ravel(),
    "sigma": sigmas_by_user.ravel(),
})
sigma_summary.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])


## 2. Estimation-Ready Objects

The EM code uses compact integer arrays:

| Array | Shape | Meaning |
|---|---:|---|
| `obs_user_idx` | observations | Integer user index for each interaction |
| `obs_cat_idx` | observations | Integer category index for each interaction |
| `obs_sess_idx` | observations | Integer session index for each interaction |
| `obs_log_watch_ratio` | observations | Measurement `s_itj` |
| `session_beliefs` | sessions by categories | Belief vector `b_isk` before the session |
| `sigmas_by_user` | users by categories | Utility standard deviation `sigma_ik` |

Rows with missing required covariates are dropped before these arrays are formed.


### Session-Level Belief Table

Notebook 05 produces one belief vector per `(user_id, burst_id, session, category)`. This cell pivots that long table to one row per session.

If a session has no valid belief mass after merging, it receives the uniform distribution over observed categories. Each belief vector is renormalized so that

$$
    \sum_k b_{isk}=1.
$$


In [ ]:
session_keys = (
    df_valid[["user_id", "burst_id", "session"]]
    .drop_duplicates()
    .copy()
)
session_keys["user_id"] = session_keys["user_id"].astype(int)
session_keys["burst_id"] = session_keys["burst_id"].astype(int)
session_keys["session"] = session_keys["session"].astype(int)
session_keys["user_idx"] = session_keys["user_id"].map(uid_to_idx).astype(int)

belief_df = belief_raw[belief_raw["user_id"].isin(user_ids)].copy()
belief_df["user_id"] = belief_df["user_id"].astype(int)
belief_df["burst_id"] = belief_df["burst_id"].astype(int)
belief_df["session"] = belief_df["session"].astype(int)
belief_df["i_cat_level1_id"] = belief_df["i_cat_level1_id"].astype(int)
belief_df["belief_prob"] = pd.to_numeric(belief_df["belief_prob"], errors="coerce").fillna(0.0)

belief_wide = (
    belief_df
    .pivot_table(
        index=["user_id", "burst_id", "session"],
        columns="i_cat_level1_id",
        values="belief_prob",
        aggfunc="first",
    )
    .reset_index()
)
belief_wide.columns.name = None

wide = session_keys.merge(
    belief_wide,
    on=["user_id", "burst_id", "session"],
    how="left",
)

for cid in cat_ids:
    if cid not in wide.columns:
        wide[cid] = 0.0

wide = wide.sort_values(["user_idx", "burst_id", "session"]).reset_index(drop=True)
wide["sess_idx"] = np.arange(len(wide), dtype=np.int64)

belief_mat = wide[list(cat_ids)].fillna(0.0).to_numpy(dtype=float)
belief_mat = np.maximum(belief_mat, 0.0)
row_sums = belief_mat.sum(axis=1, keepdims=True)
zero_belief_rows = (row_sums[:, 0] <= 1e-12)
if zero_belief_rows.any():
    belief_mat[zero_belief_rows, :] = 1.0 / K_cat
    row_sums = belief_mat.sum(axis=1, keepdims=True)

belief_mat = belief_mat / np.maximum(row_sums, 1e-12)
wide["belief_vec"] = list(belief_mat)

session_table = wide[["user_id", "user_idx", "burst_id", "session", "sess_idx", "belief_vec"]].copy()

print("estimation sessions:", session_table.shape)
print("zero-belief fallback rows:", int(zero_belief_rows.sum()))
print("belief row-sum min/max:", float(belief_mat.sum(axis=1).min()), float(belief_mat.sum(axis=1).max()))
session_table.head()

### Compact Interaction Table

Only variables used by the structural likelihood are kept here. Outcome heads are not used in the structural likelihood after the score proxy has been constructed, so the E-step cannot directly leak the four behavioral labels into the measurement equation.


In [ ]:
id_cols = ["row_id", "user_id", "video_id", "burst_id", "session"]
debug_time_cols = [c for c in ["date", "time", "timestamp"] if c in df_valid.columns]

em_feature_cols = [
    "i_cat_level1_id",
    "i_video_duration",
    "sess_rank",
    "sess_len",
    "ctx_hour_sin",
    "ctx_hour_cos",
    "ctx_is_weekend",
    "u_is_lowactive_period",
    "hist_has_author_history",
    "hist_ema_watchratio",
    "log_watch_ratio",
]

missing_feature_cols = [c for c in em_feature_cols if c not in df_valid.columns]
if missing_feature_cols:
    raise KeyError(f"df_valid is missing required EM columns: {missing_feature_cols}")

cols_keep = id_cols + debug_time_cols + em_feature_cols
interaction_df = df_valid[cols_keep].copy()

interaction_df["row_id"] = interaction_df["row_id"].astype(np.int64)
interaction_df["user_id"] = interaction_df["user_id"].astype(int)
interaction_df["burst_id"] = interaction_df["burst_id"].astype(int)
interaction_df["session"] = interaction_df["session"].astype(int)
interaction_df["i_cat_level1_id"] = interaction_df["i_cat_level1_id"].astype(int)

numeric_cols = [
    "i_video_duration",
    "sess_rank",
    "sess_len",
    "ctx_hour_sin",
    "ctx_hour_cos",
    "ctx_is_weekend",
    "u_is_lowactive_period",
    "hist_has_author_history",
    "hist_ema_watchratio",
    "log_watch_ratio",
]
for col in numeric_cols:
    interaction_df[col] = pd.to_numeric(interaction_df[col], errors="coerce")

interaction_df["user_idx_int"] = interaction_df["user_id"].map(uid_to_idx)
interaction_df["cat_idx_int"] = interaction_df["i_cat_level1_id"].map(cat_to_idx)

interaction_df = interaction_df.merge(
    session_table[["user_id", "burst_id", "session", "sess_idx"]],
    on=["user_id", "burst_id", "session"],
    how="left",
    validate="many_to_one",
)

required_nonnull = numeric_cols + ["user_idx_int", "cat_idx_int", "sess_idx"]
n_before = len(interaction_df)
interaction_df = interaction_df.dropna(subset=required_nonnull).copy()
n_after = len(interaction_df)

# If cleanup removed every interaction in a session, prune that session and reindex sess_idx.
observed_session_keys = interaction_df[["user_id", "burst_id", "session"]].drop_duplicates()
session_table = (
    session_table
    .drop(columns=["sess_idx"])
    .merge(observed_session_keys, on=["user_id", "burst_id", "session"], how="inner")
    .sort_values(["user_idx", "burst_id", "session"])
    .reset_index(drop=True)
)
session_table["sess_idx"] = np.arange(len(session_table), dtype=np.int64)

interaction_df = interaction_df.drop(columns=["sess_idx"]).merge(
    session_table[["user_id", "burst_id", "session", "sess_idx"]],
    on=["user_id", "burst_id", "session"],
    how="inner",
    validate="many_to_one",
)

interaction_df["user_idx_int"] = interaction_df["user_idx_int"].astype(np.int64)
interaction_df["cat_idx_int"] = interaction_df["cat_idx_int"].astype(np.int64)
interaction_df["sess_idx"] = interaction_df["sess_idx"].astype(np.int64)
interaction_df["sess_rank"] = interaction_df["sess_rank"].astype(np.float64)
interaction_df["sess_len"] = interaction_df["sess_len"].astype(np.float64)

print("interaction rows before/after non-null cleanup:", f"{n_before:,}", "->", f"{n_after:,}")
print("dropped rows:", f"{n_before - n_after:,}")
print("interaction columns:", interaction_df.columns.tolist())
interaction_df.head()

### Convenience Arrays and Contracts

The following checks are deliberately strict. If any index range, belief row sum, sigma positivity check, or session merge contract fails, the notebook stops before EM.


In [ ]:
obs_user_idx = interaction_df["user_idx_int"].to_numpy(dtype=np.int64, copy=False)
obs_cat_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)
obs_sess_idx = interaction_df["sess_idx"].to_numpy(dtype=np.int64, copy=False)
obs_log_watch_ratio = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
session_user_idx = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
session_beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)

idx_maps = {
    "user_ids": user_ids,
    "cat_ids": cat_ids,
    "uid_to_idx": uid_to_idx,
    "cat_to_idx": cat_to_idx,
}

estimation_inputs = {
    "interaction_df": interaction_df,
    "session_table": session_table,
    "sigmas_by_user": sigmas_by_user,
    "session_beliefs": session_beliefs,
    "obs_user_idx": obs_user_idx,
    "obs_cat_idx": obs_cat_idx,
    "obs_sess_idx": obs_sess_idx,
    "obs_log_watch_ratio": obs_log_watch_ratio,
    "idx_maps": idx_maps,
}

print("obs_user_idx:", obs_user_idx.shape, obs_user_idx.min(), obs_user_idx.max())
print("obs_cat_idx:", obs_cat_idx.shape, obs_cat_idx.min(), obs_cat_idx.max())
print("obs_sess_idx:", obs_sess_idx.shape, obs_sess_idx.min(), obs_sess_idx.max())
print("session_beliefs:", session_beliefs.shape)
print("sigmas_by_user:", sigmas_by_user.shape)

In [ ]:
checks = {
    "sigmas_shape_ok": sigmas_by_user.shape == (N_users, K_cat),
    "sigmas_finite": np.isfinite(sigmas_by_user).all(),
    "sigmas_positive": (sigmas_by_user > 0).all(),
    "belief_shape_ok": session_beliefs.shape == (len(session_table), K_cat),
    "belief_rows_sum_to_one": np.allclose(session_beliefs.sum(axis=1), 1.0),
    "obs_user_idx_in_range": obs_user_idx.min() >= 0 and obs_user_idx.max() < N_users,
    "obs_cat_idx_in_range": obs_cat_idx.min() >= 0 and obs_cat_idx.max() < K_cat,
    "obs_sess_idx_in_range": obs_sess_idx.min() >= 0 and obs_sess_idx.max() < len(session_table),
    "one_session_row_per_observed_session": interaction_df[["user_id", "burst_id", "session"]].drop_duplicates().shape[0] == len(session_table),
    "no_required_missing": not interaction_df[required_nonnull].isna().any().any(),
}

for name, ok in checks.items():
    print(f"{name}: {ok}")

if not all(checks.values()):
    failed = [name for name, ok in checks.items() if not ok]
    raise ValueError(f"Estimation input validation failed: {failed}")

## 3. Search-Threshold Solver

For each user-session, the search threshold `tau_is` solves

$$
    E\left[(U_{is}-\tau_{is})_+\right] = c_i,
$$

where the session-level utility distribution is a belief-weighted normal mixture:

$$
    U_{is}\sim \sum_k b_{isk}\,N(\mu_{ik},\sigma_{ik}^2).
$$

For a single normal component,

$$
    E[(U-\tau)_+]
    = (\mu-\tau)\left[1-\Phi\left(\frac{\tau-\mu}{\sigma}\right)\right]
      + \sigma\phi\left(\frac{\tau-\mu}{\sigma}\right).
$$

The solver uses bisection, but first it validates the bracket. At the lower endpoint the expected gain must be at least the cost; at the upper endpoint it must be at most the cost. If not, the bracket expands before bisection. The notebook records residual diagnostics after solving.


In [ ]:

from scipy.special import erf

threshold_diagnostic_records = []


def standard_normal_pdf(x):
    x = np.asarray(x, dtype=np.float64)
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * x * x)


def standard_normal_cdf(x):
    x = np.asarray(x, dtype=np.float64)
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))


def theta_to_mu_matrix(theta, n_users=None, k_cat=None, center_reference=True):
    """Return the user-category utility mean matrix, shape [N_users, K_cat]."""
    if "mu" in theta:
        mu = np.asarray(theta["mu"], dtype=np.float64)
    else:
        U = np.asarray(theta["U"], dtype=np.float64)
        V = np.asarray(theta["V"], dtype=np.float64)
        inferred_n, inferred_k = U.shape[0], V.shape[0]

        b_cat = np.asarray(theta.get("b_cat", np.zeros(inferred_k)), dtype=np.float64)
        if b_cat.ndim == 0:
            b_cat = np.full(inferred_k, float(b_cat), dtype=np.float64)

        mu = b_cat[None, :] + U @ V.T

    if n_users is not None and mu.shape[0] != int(n_users):
        raise ValueError(f"mu has {mu.shape[0]} users, expected {n_users}")
    if k_cat is not None and mu.shape[1] != int(k_cat):
        raise ValueError(f"mu has {mu.shape[1]} categories, expected {k_cat}")

    if center_reference:
        reference_cat_idx = theta.get("reference_cat_idx", globals().get("UTILITY_REFERENCE_CAT_IDX", None))
        if reference_cat_idx is not None:
            reference_cat_idx = int(reference_cat_idx)
            if reference_cat_idx < 0 or reference_cat_idx >= mu.shape[1]:
                raise ValueError(f"reference_cat_idx={reference_cat_idx} is outside K={mu.shape[1]}")
            mu = mu - mu[:, [reference_cat_idx]]

    return mu


def theta_to_cost_vector(theta, n_users, cost_floor=COST_FLOOR):
    c = np.asarray(theta.get("c", np.full(n_users, float(cost_floor))), dtype=np.float64)
    if c.ndim == 0:
        c = np.full(n_users, float(c), dtype=np.float64)
    if c.shape[0] != int(n_users):
        raise ValueError(f"c has length {c.shape[0]}, expected {n_users}")
    return np.maximum(c, float(cost_floor))


def expected_marginal_gain_mixture_vec(tau, mus, sigmas, beliefs):
    """Compute E[(U - tau)+] for many belief-weighted normal mixtures."""
    tau = np.asarray(tau, dtype=np.float64)
    if tau.ndim == 1:
        tau = tau[:, None]
    mus = np.asarray(mus, dtype=np.float64)
    sigmas = np.maximum(np.asarray(sigmas, dtype=np.float64), 1e-12)
    beliefs = np.asarray(beliefs, dtype=np.float64)

    a = (tau - mus) / sigmas
    tail = 1.0 - standard_normal_cdf(a)
    pdf_a = standard_normal_pdf(a)
    gain_by_cat = (mus - tau) * tail + sigmas * pdf_a
    return np.sum(beliefs * gain_by_cat, axis=1)


def summarize_threshold_residuals(residual, failed_count=0, bracket_expansions=0, label="threshold_solve"):
    residual = np.asarray(residual, dtype=np.float64).reshape(-1)
    abs_resid = np.abs(residual[np.isfinite(residual)])
    if abs_resid.size == 0:
        return {
            "label": label,
            "n_sessions": int(residual.size),
            "mean_abs_threshold_residual": np.nan,
            "median_abs_threshold_residual": np.nan,
            "p95_abs_threshold_residual": np.nan,
            "max_abs_threshold_residual": np.nan,
            "num_failed_thresholds": int(failed_count),
            "bracket_expansions": int(bracket_expansions),
        }
    return {
        "label": label,
        "n_sessions": int(residual.size),
        "mean_abs_threshold_residual": float(np.mean(abs_resid)),
        "median_abs_threshold_residual": float(np.median(abs_resid)),
        "p95_abs_threshold_residual": float(np.percentile(abs_resid, 95)),
        "max_abs_threshold_residual": float(np.max(abs_resid)),
        "num_failed_thresholds": int(failed_count),
        "bracket_expansions": int(bracket_expansions),
    }


def solve_tau_vectorized(
    mus,
    sigmas,
    beliefs,
    costs,
    max_iter=60,
    bracket_width=10.0,
    max_bracket_expansions=20,
    cost_floor=COST_FLOOR,
    return_diagnostics=False,
    diagnostic_label="threshold_solve",
    raise_on_bracket_failure=True,
):
    """Solve E[(U - tau)+] = c using validated, expanding vectorized bisection."""
    mus = np.asarray(mus, dtype=np.float64)
    sigmas = np.maximum(np.asarray(sigmas, dtype=np.float64), 1e-12)
    beliefs = np.asarray(beliefs, dtype=np.float64)
    costs = np.asarray(costs, dtype=np.float64).reshape(-1)

    if mus.shape != sigmas.shape or mus.shape != beliefs.shape:
        raise ValueError("mus, sigmas, and beliefs must have identical shape [S, K]")
    if costs.shape[0] != mus.shape[0]:
        raise ValueError("costs must have length S")

    belief_sums = beliefs.sum(axis=1, keepdims=True)
    beliefs = beliefs / np.maximum(belief_sums, 1e-12)
    costs = np.maximum(costs, float(cost_floor))

    smax = np.maximum(np.max(sigmas, axis=1), 1e-8)
    span = float(bracket_width) * smax
    lo = np.min(mus, axis=1) - span
    hi = np.max(mus, axis=1) + span

    expansions = 0
    for expansion in range(int(max_bracket_expansions) + 1):
        gain_lo = expected_marginal_gain_mixture_vec(lo, mus, sigmas, beliefs)
        gain_hi = expected_marginal_gain_mixture_vec(hi, mus, sigmas, beliefs)
        bad_lo = gain_lo < costs
        bad_hi = gain_hi > costs
        if not (bad_lo.any() or bad_hi.any()):
            break
        if expansion == int(max_bracket_expansions):
            failed = int(bad_lo.sum() + bad_hi.sum())
            diag = summarize_threshold_residuals(
                np.full_like(costs, np.nan),
                failed_count=failed,
                bracket_expansions=expansions,
                label=diagnostic_label,
            )
            msg = (
                f"Threshold bracket failed for {failed} endpoint checks after "
                f"{max_bracket_expansions} expansions. "
                f"bad_lo={int(bad_lo.sum())}, bad_hi={int(bad_hi.sum())}, "
                f"label={diagnostic_label!r}"
            )
            if raise_on_bracket_failure:
                raise ValueError(msg)
            if return_diagnostics:
                return np.full_like(costs, np.nan), diag
            raise ValueError(msg)
        expand_span = span * (2.0 ** expansion)
        lo = np.where(bad_lo, lo - expand_span, lo)
        hi = np.where(bad_hi, hi + expand_span, hi)
        expansions += 1

    for _ in range(int(max_iter)):
        mid = 0.5 * (lo + hi)
        gain = expected_marginal_gain_mixture_vec(mid, mus, sigmas, beliefs)
        should_raise_tau = gain > costs
        lo = np.where(should_raise_tau, mid, lo)
        hi = np.where(should_raise_tau, hi, mid)

    tau = 0.5 * (lo + hi)
    final_gain = expected_marginal_gain_mixture_vec(tau, mus, sigmas, beliefs)
    residual = final_gain - costs
    diag = summarize_threshold_residuals(residual, failed_count=0, bracket_expansions=expansions, label=diagnostic_label)
    if return_diagnostics:
        return tau, diag
    return tau


def solve_tau_for_sessions(session_table, theta, sigmas_by_user, max_iter=60, return_diagnostics=False, diagnostic_label="sessions"):
    """Solve one threshold per row of `session_table`."""
    mu_matrix = theta_to_mu_matrix(theta, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
    cost_vector = theta_to_cost_vector(theta, n_users=sigmas_by_user.shape[0])

    sess_users = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
    beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)

    session_mus = mu_matrix[sess_users]
    session_sigmas = sigmas_by_user[sess_users]
    session_costs = cost_vector[sess_users]

    return solve_tau_vectorized(
        session_mus,
        session_sigmas,
        beliefs,
        session_costs,
        max_iter=max_iter,
        return_diagnostics=return_diagnostics,
        diagnostic_label=diagnostic_label,
    )


In [ ]:

# Smoke tests for the threshold solver.
mus_test = np.array([[0.0, 1.0]], dtype=np.float64)
sigmas_test = np.array([[1.0, 1.0]], dtype=np.float64)
beliefs_test = np.array([[0.5, 0.5]], dtype=np.float64)
costs_test = np.array([0.1], dtype=np.float64)

tau_test, tau_diag_test = solve_tau_vectorized(
    mus_test,
    sigmas_test,
    beliefs_test,
    costs_test,
    return_diagnostics=True,
    diagnostic_label="threshold_unit_finite",
)
assert np.isfinite(tau_test).all()
assert tau_diag_test["max_abs_threshold_residual"] < THRESHOLD_RESIDUAL_TOL

try:
    solve_tau_vectorized(
        mus_test,
        sigmas_test,
        beliefs_test,
        costs_test,
        bracket_width=1e-12,
        max_bracket_expansions=0,
        return_diagnostics=True,
        diagnostic_label="threshold_unit_raise",
    )
except ValueError:
    pass
else:
    raise AssertionError("Expected threshold solver to raise on bracket failure.")

# Smoke test with a small random theta. This checks shapes and finite output on real session objects.
rng = np.random.default_rng(0)
theta_smoke = {
    "b_cat": rng.normal(0.0, 0.2, size=K_cat),
    "U": rng.normal(0.0, 0.05, size=(N_users, 4)),
    "V": rng.normal(0.0, 0.05, size=(K_cat, 4)),
    "c": np.full(N_users, 0.15),
}

tau_smoke = solve_tau_for_sessions(session_table.head(1000), theta_smoke, sigmas_by_user)
assert np.isfinite(tau_smoke).all()
print("tau smoke shape:", tau_smoke.shape)
print("tau smoke range:", float(tau_smoke.min()), float(tau_smoke.max()))
print("threshold solver smoke tests passed")


## 4. Watch-Ratio Measurement Model

The measurement model links latent state `z_itj` to observed log watch ratio:

$$
    s_{itj}\mid z_{itj}=z \sim N(m_{itj,z}, v_{itj,z}).
$$

The state-specific mean is

$$
    m_{itj,z}=\alpha^{\mu}_{k,z}+x^{\mu}_{itj}\beta^{\mu}_z.
$$

The state-specific variance is

$$
    v_{itj,z}=\exp(\eta_{itj,z})+v_{min},
    \qquad
    \eta_{itj,z}=\alpha^{v}_{k,z}+x^v_{itj}\beta^v_z.
$$

For numerical stability the code never evaluates `log(exp(eta)+v_min)` by exponentiating `eta` directly. It uses

$$
    \log v_{itj,z}=\operatorname{logaddexp}(\eta_{itj,z}, \log v_{min}).
$$

The Gaussian log density is evaluated as

$$
    \log f_z(s_{itj}) = -\frac12\left[\log(2\pi)+\log v_{itj,z}+\frac{(s_{itj}-m_{itj,z})^2}{v_{itj,z}}\right].
$$


In [ ]:

def build_watch_feature_blocks(interaction_df):
    """Build feature matrices for the watch-ratio measurement model."""
    df = interaction_df
    required = [
        "i_video_duration", "sess_rank", "sess_len",
        "ctx_hour_sin", "ctx_hour_cos", "ctx_is_weekend",
        "u_is_lowactive_period",
        "hist_has_author_history", "hist_ema_watchratio",
    ]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"interaction_df missing watch-model columns: {missing}")

    duration = df["i_video_duration"].to_numpy(dtype=np.float32, copy=False)
    log_duration = np.log1p(np.maximum(duration, 0.0)).astype(np.float32)

    sess_rank = df["sess_rank"].to_numpy(dtype=np.float32, copy=False)
    sess_len = df["sess_len"].to_numpy(dtype=np.float32, copy=False)
    rel_rank = (sess_rank / np.maximum(sess_len, 1.0)).astype(np.float32)
    rel_rank_sq = (rel_rank * rel_rank).astype(np.float32)

    hour_sin = df["ctx_hour_sin"].to_numpy(dtype=np.float32, copy=False)
    hour_cos = df["ctx_hour_cos"].to_numpy(dtype=np.float32, copy=False)
    weekend = df["ctx_is_weekend"].to_numpy(dtype=np.float32, copy=False)
    low_activity = df["u_is_lowactive_period"].to_numpy(dtype=np.float32, copy=False)
    has_author_history = df["hist_has_author_history"].to_numpy(dtype=np.float32, copy=False)
    ema_watchratio = df["hist_ema_watchratio"].to_numpy(dtype=np.float32, copy=False)

    X_mean = np.vstack([
        log_duration,
        rel_rank,
        rel_rank_sq,
        hour_sin,
        hour_cos,
        weekend,
        low_activity,
        has_author_history,
        ema_watchratio,
    ]).T.astype(np.float32, copy=False)
    mean_names = [
        "log1p_i_video_duration",
        "sess_rank_over_sess_len",
        "sess_rank_over_sess_len_sq",
        "ctx_hour_sin",
        "ctx_hour_cos",
        "ctx_is_weekend",
        "u_is_lowactive_period",
        "hist_has_author_history",
        "hist_ema_watchratio",
    ]

    X_logvar = np.vstack([
        log_duration,
        rel_rank,
        hour_sin,
        hour_cos,
        weekend,
    ]).T.astype(np.float32, copy=False)
    logvar_names = [
        "log1p_i_video_duration",
        "sess_rank_over_sess_len",
        "ctx_hour_sin",
        "ctx_hour_cos",
        "ctx_is_weekend",
    ]

    meta = {
        "feature_names_mean": mean_names,
        "feature_names_logvar": logvar_names,
    }
    return X_mean, X_logvar, meta


def stable_log_variance_from_eta(eta, min_var=1e-4):
    eta = np.asarray(eta, dtype=np.float64)
    return np.logaddexp(eta, np.log(float(min_var)))


def init_phi_from_interactions(
    interaction_df,
    cat_ids,
    wr_col="log_watch_ratio",
    var_floor=1e-4,
    mean_sep=None,
    logvar_sep=0.20,
    jitter_mu=0.02,
    jitter_logvar=0.02,
    seed=0,
):
    """Data-informed initialization for the measurement parameters phi."""
    if wr_col not in interaction_df.columns:
        raise KeyError(f"interaction_df missing {wr_col}")

    cat_ids = np.asarray(cat_ids)
    K = len(cat_ids)
    stats = (
        interaction_df
        .groupby("i_cat_level1_id", observed=True)[wr_col]
        .agg(["mean", "var"])
        .reindex(cat_ids)
    )

    y = interaction_df[wr_col].to_numpy(dtype=np.float64, copy=False)
    global_mean = float(np.nanmean(y))
    global_var = max(float(np.nanvar(y)), float(var_floor))
    global_std = float(np.sqrt(global_var))
    if mean_sep is None:
        mean_sep = 0.10 * global_std

    mean_k = stats["mean"].to_numpy(dtype=np.float64)
    var_k = stats["var"].to_numpy(dtype=np.float64)
    mean_k = np.where(np.isfinite(mean_k), mean_k, global_mean)
    var_k = np.where(np.isfinite(var_k), var_k, global_var)
    var_k = np.maximum(var_k, float(var_floor))
    # eta is initialized near log(var - min_var), but log(var) is a safe approximation here.
    logvar_k = np.log(var_k)

    rng = np.random.default_rng(seed)
    alpha_mu0 = mean_k - 0.5 * mean_sep + rng.normal(0.0, jitter_mu, size=K)
    alpha_mu1 = mean_k + 0.5 * mean_sep + rng.normal(0.0, jitter_mu, size=K)
    alpha_logvar0 = logvar_k - 0.5 * logvar_sep + rng.normal(0.0, jitter_logvar, size=K)
    alpha_logvar1 = logvar_k + 0.5 * logvar_sep + rng.normal(0.0, jitter_logvar, size=K)

    X_mean, X_logvar, meta = build_watch_feature_blocks(interaction_df)

    return {
        "cat_ids": cat_ids.astype(int),
        "cat_id_to_idx": {int(cid): idx for idx, cid in enumerate(cat_ids)},
        "alpha_mu0": alpha_mu0.astype(np.float64),
        "alpha_mu1": alpha_mu1.astype(np.float64),
        "beta_mu0": np.zeros(X_mean.shape[1], dtype=np.float64),
        "beta_mu1": np.zeros(X_mean.shape[1], dtype=np.float64),
        "alpha_logvar0": alpha_logvar0.astype(np.float64),
        "alpha_logvar1": alpha_logvar1.astype(np.float64),
        "beta_logvar0": np.zeros(X_logvar.shape[1], dtype=np.float64),
        "beta_logvar1": np.zeros(X_logvar.shape[1], dtype=np.float64),
        "feature_names_mean": meta["feature_names_mean"],
        "feature_names_logvar": meta["feature_names_logvar"],
    }


def predict_watch_moments(interaction_df, phi, min_var=1e-4, return_logvar=False):
    """Return state-specific means and variances/log-variances: (m0, v0/logv0, m1, v1/logv1)."""
    X_mean, X_logvar, _ = build_watch_feature_blocks(interaction_df)
    cat_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)

    m0 = phi["alpha_mu0"][cat_idx] + X_mean @ phi["beta_mu0"]
    m1 = phi["alpha_mu1"][cat_idx] + X_mean @ phi["beta_mu1"]
    eta0 = phi["alpha_logvar0"][cat_idx] + X_logvar @ phi["beta_logvar0"]
    eta1 = phi["alpha_logvar1"][cat_idx] + X_logvar @ phi["beta_logvar1"]

    logv0 = stable_log_variance_from_eta(eta0, min_var=min_var)
    logv1 = stable_log_variance_from_eta(eta1, min_var=min_var)
    if return_logvar:
        return m0, logv0, m1, logv1
    return m0, np.exp(np.minimum(logv0, 700.0)), m1, np.exp(np.minimum(logv1, 700.0))


def log_normal_pdf_from_logvar(x, mean, log_var):
    x = np.asarray(x, dtype=np.float64)
    mean = np.asarray(mean, dtype=np.float64)
    log_var = np.asarray(log_var, dtype=np.float64)
    inv_var = np.exp(-log_var)
    return -0.5 * (np.log(2.0 * np.pi) + log_var + (x - mean) ** 2 * inv_var)


def log_normal_pdf(x, mean, var):
    var = np.maximum(np.asarray(var, dtype=np.float64), 1e-300)
    return log_normal_pdf_from_logvar(x, mean, np.log(var))


def compute_watch_loglik_given_responsibilities(interaction_df, phi, responsibilities):
    """Expected complete-data log likelihood contribution for the watch model."""
    r = np.asarray(responsibilities, dtype=np.float64).reshape(-1)
    s = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
    m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, return_logvar=True)
    return float(np.sum(r * log_normal_pdf_from_logvar(s, m1, logv1) + (1.0 - r) * log_normal_pdf_from_logvar(s, m0, logv0)))


In [ ]:
# Measurement-model smoke test on a small slice.
smoke_interactions = interaction_df.head(50_000).copy()
phi0 = init_phi_from_interactions(smoke_interactions, cat_ids=cat_ids, seed=0)
m0, v0, m1, v1 = predict_watch_moments(smoke_interactions, phi0)

print("X_mean features:", phi0["feature_names_mean"])
print("X_logvar features:", phi0["feature_names_logvar"])
print("phi category count:", len(phi0["cat_ids"]))
print("moments finite:", all(np.isfinite(arr).all() for arr in [m0, v0, m1, v1]))
print("m0/m1 head:", m0[:5], m1[:5])
print("v0/v1 head:", v0[:5], v1[:5])

## 5. EM Estimation: Low-Rank Utility Benchmark

The utility mean uses a low-rank user-category representation:

$$
    \mu_{ik}=b_k + U_i^\top V_k.
$$

Because each user's utility location is not identified, `theta_to_mu_matrix` subtracts the UNKNOWN category from every category for the same user:

$$
    \mu^{centered}_{ik}=\mu_{ik}-\mu_{iu}.
$$

The E-step computes

$$
    r_{itj}=Pr(z_{itj}=1\mid s_{itj},\theta,\phi)
    = \frac{\pi_{itj} f_1(s_{itj})}{\pi_{itj} f_1(s_{itj})+(1-\pi_{itj})f_0(s_{itj})},
$$

where

$$
    \pi_{itj}=Pr(z_{itj}=1\mid\theta)=\Phi\left(\frac{\mu_{ik}-\tau_{is}}{\sigma_{ik}}\right).
$$

The `phi` update is an M-step for the measurement model. The log-variance block minimizes

$$
    \frac12\sum_n w_n\left[\log v_n + \frac{(s_n-m_n)^2}{v_n}\right]
    + \frac{\lambda}{2}\lVert\gamma\rVert_2^2,
    \qquad v_n=\exp(\eta_n)+v_{min}.
$$

The `theta` update is an approximate fixed-threshold update. After proposing it, the notebook recomputes thresholds and accepts the proposal only if the true observed likelihood is not below the post-`phi` likelihood beyond `GEM_TOL`.


In [ ]:
from time import time

from scipy.optimize import minimize
import torch


LATENT_DIM = 15
INITIAL_SEARCH_COST = INITIAL_SEARCH_COST_RAW

# Starting regularization choices. These are intentionally mild and should be tuned by sensitivity checks.
THETA_L2_REG = 1e-5
SEARCH_COST_LOG_SHRINKAGE = 0.10
SEARCH_COST_SHRINK_TARGET = "mean"

# Anchor structural utility means to a sparse-shrunk normalized score proxy.
SCORE_ANCHOR_L2 = 0.05
SCORE_ANCHOR_N0 = 50.0
SCORE_ANCHOR_MIN_N = 1
SPARSE_SCORE_ANCHOR_MIN_WEIGHT = 0.25
UNOBSERVED_SCORE_ANCHOR_WEIGHT = 0.25


def init_theta_low_rank_from_scores(
    user_cat_score_moments,
    user_ids,
    cat_ids,
    d=8,
    initial_search_cost=0.5,
):
    """Initialize low-rank utility parameters from user-category mean score proxies."""
    score_wide = (
        user_cat_score_moments
        .pivot(index="user_id", columns="i_cat_level1_id", values="mean_score")
        .reindex(index=user_ids, columns=cat_ids)
    )
    M = score_wide.to_numpy(dtype=np.float64)

    global_mean = float(np.nanmean(M))
    user_mean = np.nanmean(M, axis=1)
    cat_mean = np.nanmean(M, axis=0)
    user_mean = np.where(np.isfinite(user_mean), user_mean, global_mean)
    cat_mean = np.where(np.isfinite(cat_mean), cat_mean, global_mean)

    cat_ids_arr = np.asarray(cat_ids)
    ref_matches = np.flatnonzero(cat_ids_arr == SCORE_REFERENCE_CATEGORY_ID)
    reference_cat_idx = int(ref_matches[0]) if len(ref_matches) else int(globals().get("UTILITY_REFERENCE_CAT_IDX", 0))

    additive_fill = user_mean[:, None] + (cat_mean[None, :] - global_mean)
    M_filled = np.where(np.isfinite(M), M, additive_fill)
    M_centered = M_filled - M_filled[:, [reference_cat_idx]]

    b_cat = np.nanmean(M_centered, axis=0).astype(np.float64)
    b_cat = np.where(np.isfinite(b_cat), b_cat, 0.0)
    b_cat = b_cat - b_cat[reference_cat_idx]

    residual = M_centered - b_cat[None, :]
    residual = np.nan_to_num(residual, nan=0.0, posinf=0.0, neginf=0.0)
    residual = residual - residual.mean(axis=0, keepdims=True)

    U_svd, s_svd, Vt_svd = np.linalg.svd(residual, full_matrices=False)
    d_eff = int(min(d, len(s_svd)))
    scale = np.sqrt(np.maximum(s_svd[:d_eff], 0.0))
    U = U_svd[:, :d_eff] * scale[None, :]
    V = Vt_svd[:d_eff, :].T * scale[None, :]

    if d_eff < d:
        U = np.pad(U, ((0, 0), (0, d - d_eff)))
        V = np.pad(V, ((0, 0), (0, d - d_eff)))

    return {
        "b_cat": b_cat,
        "U": U.astype(np.float64),
        "V": V.astype(np.float64),
        "c": np.full(len(user_ids), float(initial_search_cost), dtype=np.float64),
    }

def build_score_anchor_inputs(
    user_cat_score_moments,
    user_ids,
    cat_ids,
    min_n=1,
    n0=50.0,
    sparse_min_weight=0.25,
    unobserved_weight=0.25,
):
    """Build sparse-shrunk targets and weights for score-anchored utility regularization."""
    score_wide = (
        user_cat_score_moments
        .pivot(index="user_id", columns="i_cat_level1_id", values="mean_score")
        .reindex(index=user_ids, columns=cat_ids)
    )
    n_wide = (
        user_cat_score_moments
        .pivot(index="user_id", columns="i_cat_level1_id", values="n_score")
        .reindex(index=user_ids, columns=cat_ids)
        .fillna(0.0)
    )

    raw_target = score_wide.to_numpy(dtype=np.float64)
    n_obs_anchor = n_wide.to_numpy(dtype=np.float64)
    observed = np.isfinite(raw_target) & (n_obs_anchor >= float(min_n))

    reliability = n_obs_anchor / (n_obs_anchor + float(n0))
    target = np.where(observed, reliability * raw_target, 0.0)

    weights_observed = np.maximum(reliability, float(sparse_min_weight))
    weights = np.where(observed, weights_observed, float(unobserved_weight))

    return target.astype(np.float64), weights.astype(np.float64)


theta_init = init_theta_low_rank_from_scores(
    user_cat_score_moments=user_cat_score_moments,
    user_ids=user_ids,
    cat_ids=cat_ids,
    d=LATENT_DIM,
    initial_search_cost=INITIAL_SEARCH_COST,
)
theta_init["reference_cat_idx"] = UTILITY_REFERENCE_CAT_IDX
theta_init["c"] = initial_cost_by_user.copy()

score_anchor_matrix, score_anchor_weight = build_score_anchor_inputs(
    user_cat_score_moments=user_cat_score_moments,
    user_ids=user_ids,
    cat_ids=cat_ids,
    min_n=SCORE_ANCHOR_MIN_N,
    n0=SCORE_ANCHOR_N0,
    sparse_min_weight=SPARSE_SCORE_ANCHOR_MIN_WEIGHT,
    unobserved_weight=UNOBSERVED_SCORE_ANCHOR_WEIGHT,
)

phi_init = init_phi_from_interactions(interaction_df, cat_ids=cat_ids, seed=0)

theta_param_count = (
    theta_init["b_cat"].size
    + theta_init["U"].size
    + theta_init["V"].size
    + theta_init["c"].size
)
phi_param_count = sum(
    np.asarray(phi_init[name]).size
    for name in [
        "alpha_mu0", "alpha_mu1", "beta_mu0", "beta_mu1",
        "alpha_logvar0", "alpha_logvar1", "beta_logvar0", "beta_logvar1",
    ]
)

print("theta parameter count:", theta_param_count)
print("phi parameter count:", phi_param_count)
print("theta mu range:", np.percentile(theta_to_mu_matrix(theta_init), [0, 5, 50, 95, 100]))
print("initial c after per-user scale normalization:", np.percentile(theta_init["c"], [0, 50, 100]))
print("score-anchor weighted cells:", int((score_anchor_weight > 0).sum()))
print("score-anchor weight percentiles:", np.percentile(score_anchor_weight[score_anchor_weight > 0], [1, 50, 99]))
print("sparse score anchor min weight:", SPARSE_SCORE_ANCHOR_MIN_WEIGHT)
print("unobserved neutral anchor weight:", UNOBSERVED_SCORE_ANCHOR_WEIGHT)
print("reference mu check:", np.abs(theta_to_mu_matrix(theta_init)[:, UTILITY_REFERENCE_CAT_IDX]).max())

In [ ]:

def compute_e_step_with_ll(
    interaction_df,
    session_table,
    theta,
    phi,
    sigmas_by_user,
    return_session_tau=False,
    return_threshold_diagnostics=False,
    diagnostic_label="e_step",
):
    """E-step: posterior responsibilities, interaction thresholds, and observed log likelihood."""
    mu_matrix = theta_to_mu_matrix(theta, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
    cost_vector = theta_to_cost_vector(theta, n_users=sigmas_by_user.shape[0])

    sess_users = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
    beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)
    tau_sessions, tau_diag = solve_tau_vectorized(
        mu_matrix[sess_users],
        sigmas_by_user[sess_users],
        beliefs,
        cost_vector[sess_users],
        return_diagnostics=True,
        diagnostic_label=diagnostic_label,
    )
    threshold_diagnostic_records.append(dict(tau_diag))

    if not np.isfinite(tau_sessions).all():
        bad = int((~np.isfinite(tau_sessions)).sum())
        raise RuntimeError(f"Non-finite tau_sessions in E-step: {bad} bad values; label={diagnostic_label!r}")

    max_abs_resid = tau_diag.get("max_abs_threshold_residual", np.nan)
    if np.isfinite(max_abs_resid) and max_abs_resid > THRESHOLD_RESIDUAL_TOL:
        raise RuntimeError(
            f"Threshold residual too large in E-step: max_abs_residual={max_abs_resid}, "
            f"tol={THRESHOLD_RESIDUAL_TOL}, label={diagnostic_label!r}"
        )

    u_idx = interaction_df["user_idx_int"].to_numpy(dtype=np.int64, copy=False)
    k_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)
    s_idx = interaction_df["sess_idx"].to_numpy(dtype=np.int64, copy=False)
    tau_i = tau_sessions[s_idx]

    mu_i = mu_matrix[u_idx, k_idx]
    sigma_i = np.maximum(sigmas_by_user[u_idx, k_idx], 1e-8)
    z_search = (mu_i - tau_i) / sigma_i

    pi_raw = standard_normal_cdf(z_search)
    if not np.isfinite(pi_raw).all():
        bad = int((~np.isfinite(pi_raw)).sum())
        raise RuntimeError(f"Non-finite pi in E-step: {bad} bad values; label={diagnostic_label!r}")
    raw_min = float(np.nanmin(pi_raw))
    raw_max = float(np.nanmax(pi_raw))
    if raw_min < -1e-12 or raw_max > 1.0 + 1e-12:
        raise RuntimeError(f"pi outside [0,1] before clipping; min={raw_min}, max={raw_max}, label={diagnostic_label!r}")

    pi = np.clip(pi_raw, 1e-12, 1.0 - 1e-12)
    if not np.isfinite(pi).all():
        bad = int((~np.isfinite(pi)).sum())
        raise RuntimeError(f"Non-finite clipped pi in E-step: {bad} bad values; label={diagnostic_label!r}")
    clipped_min = float(np.nanmin(pi))
    clipped_max = float(np.nanmax(pi))
    if clipped_min < 0.0 or clipped_max > 1.0:
        raise RuntimeError(f"pi outside [0,1] after clipping; min={clipped_min}, max={clipped_max}, label={diagnostic_label!r}")

    log_pi = np.log(pi)
    log_one_minus_pi = np.log1p(-pi)

    y = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
    m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, return_logvar=True)
    log_f0 = log_normal_pdf_from_logvar(y, m0, logv0)
    log_f1 = log_normal_pdf_from_logvar(y, m1, logv1)

    log_joint1 = log_pi + log_f1
    log_joint0 = log_one_minus_pi + log_f0
    log_lik_i = np.logaddexp(log_joint1, log_joint0)
    total_ll = float(np.sum(log_lik_i))
    if not np.isfinite(total_ll):
        raise RuntimeError(f"Non-finite observed log likelihood in E-step: ll={total_ll}; label={diagnostic_label!r}")

    r_i = np.exp(log_joint1 - log_lik_i)
    if not np.isfinite(r_i).all():
        bad = int((~np.isfinite(r_i)).sum())
        raise RuntimeError(f"Non-finite responsibilities in E-step: {bad} bad values; label={diagnostic_label!r}")
    r_min = float(np.nanmin(r_i))
    r_max = float(np.nanmax(r_i))
    if r_min < -1e-8 or r_max > 1.0 + 1e-8:
        raise RuntimeError(f"Responsibilities outside [0,1]: min={r_min}, max={r_max}, label={diagnostic_label!r}")

    outputs = [r_i, tau_i, total_ll]
    if return_session_tau:
        outputs.append(tau_sessions)
    if return_threshold_diagnostics:
        outputs.append(tau_diag)
    return tuple(outputs) if len(outputs) > 3 else (r_i, tau_i, total_ll)


r0, tau_i0, ll0, tau_s0, tau_diag0 = compute_e_step_with_ll(
    interaction_df.head(50_000),
    session_table,
    theta_init,
    phi_init,
    sigmas_by_user,
    return_session_tau=True,
    return_threshold_diagnostics=True,
    diagnostic_label="e_step_smoke",
)
print("E-step smoke responsibilities:", r0.shape, float(r0.min()), float(r0.mean()), float(r0.max()))
print("E-step smoke LL:", ll0)
print("E-step smoke threshold diagnostics:")
print(tau_diag0)


In [ ]:

from scipy.special import expit

logvar_optimizer_diagnostic_records = []


def solve_wls_with_category_fe(X, y, weights, cat_idx, k_cat, ridge=1e-8):
    """Weighted least squares with category fixed effects and dense covariates."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    weights = np.asarray(weights, dtype=np.float64).reshape(-1)
    cat_idx = np.asarray(cat_idx, dtype=np.int64).reshape(-1)

    n_obs, p = X.shape
    if n_obs == 0 or weights.sum() <= 1e-12:
        return np.zeros(k_cat, dtype=np.float64), np.zeros(p, dtype=np.float64)

    sum_wk = np.bincount(cat_idx, weights=weights, minlength=k_cat).astype(np.float64)
    sum_wk = np.maximum(sum_wk, 1e-12)
    inv_sum_wk = 1.0 / sum_wk
    d = np.bincount(cat_idx, weights=weights * y, minlength=k_cat).astype(np.float64)

    WX = X * weights[:, None]
    E = X.T @ WX
    E[np.arange(p), np.arange(p)] += float(ridge)
    f = X.T @ (weights * y)

    B = np.empty((k_cat, p), dtype=np.float64)
    for j in range(p):
        B[:, j] = np.bincount(cat_idx, weights=weights * X[:, j], minlength=k_cat)

    B_scaled = B * inv_sum_wk[:, None]
    S = E - B.T @ B_scaled
    rhs = f - B.T @ (inv_sum_wk * d)

    try:
        beta = np.linalg.solve(S, rhs)
    except np.linalg.LinAlgError:
        beta = np.linalg.lstsq(S, rhs, rcond=None)[0]
    alpha = inv_sum_wk * (d - B @ beta)
    return alpha, beta


def optimize_logvar_block(
    X_logvar,
    resid,
    weights,
    cat_idx,
    k_cat,
    alpha_init,
    beta_init,
    min_var=1e-4,
    ridge=1e-6,
    maxiter=25,
    diagnostic_label="logvar",
):
    """Optimize one state's weighted Gaussian log-variance block with stable logaddexp math."""
    X = np.asarray(X_logvar, dtype=np.float64)
    resid2 = np.asarray(resid, dtype=np.float64) ** 2
    weights = np.asarray(weights, dtype=np.float64)
    cat_idx = np.asarray(cat_idx, dtype=np.int64)
    p = X.shape[1]
    log_min_var = np.log(float(min_var))

    x0 = np.concatenate([np.asarray(alpha_init, dtype=np.float64), np.asarray(beta_init, dtype=np.float64)])

    def obj_grad(params):
        alpha = params[:k_cat]
        beta = params[k_cat:]
        eta = alpha[cat_idx] + X @ beta
        log_var = np.logaddexp(eta, log_min_var)
        inv_var = np.exp(-log_var)
        obj_terms = 0.5 * weights * (log_var + resid2 * inv_var)
        obj = float(np.sum(obj_terms) + 0.5 * float(ridge) * np.sum(params * params))

        dlogvar_deta = expit(eta - log_min_var)
        d_obj_d_eta = 0.5 * weights * dlogvar_deta * (1.0 - resid2 * inv_var)
        grad_alpha = np.bincount(cat_idx, weights=d_obj_d_eta, minlength=k_cat)
        grad_beta = X.T @ d_obj_d_eta
        grad = np.concatenate([grad_alpha, grad_beta]) + float(ridge) * params
        return obj, grad

    initial_obj, initial_grad = obj_grad(x0)
    result = minimize(
        fun=lambda p: obj_grad(p)[0],
        x0=x0,
        jac=lambda p: obj_grad(p)[1],
        method="L-BFGS-B",
        options={"maxiter": int(maxiter), "ftol": 1e-8, "maxls": 20},
    )
    final_obj, final_grad = obj_grad(result.x)
    logvar_optimizer_diagnostic_records.append({
        "label": diagnostic_label,
        "success": bool(result.success),
        "status": int(result.status),
        "message": str(result.message),
        "nit": int(getattr(result, "nit", -1)),
        "nfev": int(getattr(result, "nfev", -1)),
        "initial_obj": float(initial_obj),
        "final_obj": float(final_obj),
        "initial_grad_norm": float(np.linalg.norm(initial_grad)),
        "final_grad_norm": float(np.linalg.norm(final_grad)),
    })
    params = result.x
    return params[:k_cat], params[k_cat:], result


def update_phi_coordinate_descent(
    interaction_df,
    responsibilities,
    phi_old,
    n_coord_iters=1,
    min_var=1e-4,
    ridge_mean=1e-6,
    ridge_logvar=1e-6,
    logvar_maxiter=20,
):
    """M-step for phi by alternating weighted mean and log-variance updates."""
    phi = {key: (val.copy() if isinstance(val, np.ndarray) else val) for key, val in phi_old.items()}
    y = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
    r = np.asarray(responsibilities, dtype=np.float64).reshape(-1)
    cat_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)
    k_cat = int(len(phi["cat_ids"]))
    X_mean, X_logvar, _ = build_watch_feature_blocks(interaction_df)
    X_mean = X_mean.astype(np.float64)
    X_logvar = X_logvar.astype(np.float64)

    for coord_iter in range(int(n_coord_iters)):
        m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, min_var=min_var, return_logvar=True)
        for state, weights_state, logvar_state in [
            (0, 1.0 - r, logv0),
            (1, r, logv1),
        ]:
            w_mean = weights_state * np.exp(-logvar_state)
            alpha_mu, beta_mu = solve_wls_with_category_fe(
                X_mean, y, w_mean, cat_idx, k_cat, ridge=ridge_mean
            )
            phi[f"alpha_mu{state}"] = alpha_mu
            phi[f"beta_mu{state}"] = beta_mu

        m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, min_var=min_var, return_logvar=True)
        for state, weights_state, mean_state in [
            (0, 1.0 - r, m0),
            (1, r, m1),
        ]:
            alpha_lv, beta_lv, _ = optimize_logvar_block(
                X_logvar=X_logvar,
                resid=y - mean_state,
                weights=weights_state,
                cat_idx=cat_idx,
                k_cat=k_cat,
                alpha_init=phi[f"alpha_logvar{state}"],
                beta_init=phi[f"beta_logvar{state}"],
                min_var=min_var,
                ridge=ridge_logvar,
                maxiter=logvar_maxiter,
                diagnostic_label=f"coord{coord_iter}_state{state}",
            )
            phi[f"alpha_logvar{state}"] = alpha_lv
            phi[f"beta_logvar{state}"] = beta_lv

    return phi


def swap_phi_states(phi):
    """Swap the measurement-model labels z=0 and z=1."""
    out = {key: (val.copy() if isinstance(val, np.ndarray) else val) for key, val in phi.items()}
    for stem in ["alpha_mu", "beta_mu", "alpha_logvar", "beta_logvar"]:
        key0 = f"{stem}0"
        key1 = f"{stem}1"
        out[key0], out[key1] = out[key1].copy(), out[key0].copy()
    return out


def orient_phi_high_watch_state(interaction_df, phi, sample_size=200_000, seed=0):
    """Ensure z=1 is the higher predicted log-watch-ratio state."""
    if sample_size is not None and len(interaction_df) > int(sample_size):
        check_df = interaction_df.sample(n=int(sample_size), random_state=int(seed))
    else:
        check_df = interaction_df

    m0, _, m1, _ = predict_watch_moments(check_df, phi)
    mean_m0 = float(np.mean(m0))
    mean_m1 = float(np.mean(m1))
    share_m1_gt_m0 = float(np.mean(m1 > m0))
    swapped = mean_m1 < mean_m0

    if swapped:
        phi = swap_phi_states(phi)
        mean_m0, mean_m1 = mean_m1, mean_m0
        share_m1_gt_m0 = 1.0 - share_m1_gt_m0

    info = {
        "phi_state_swapped": bool(swapped),
        "phi_mean_m0": mean_m0,
        "phi_mean_m1": mean_m1,
        "phi_mean_m1_minus_m0": mean_m1 - mean_m0,
        "phi_share_m1_gt_m0": share_m1_gt_m0,
    }
    return phi, info


In [ ]:

def update_search_costs_from_thresholds(
    session_table,
    theta_without_c,
    sigmas_by_user,
    tau_sessions,
    c_floor=COST_FLOOR,
    log_shrinkage=0.10,
    shrink_target="mean",
):
    """Update c_i from the threshold equation, with optional log-space shrinkage."""
    mu_matrix = theta_to_mu_matrix(theta_without_c, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
    sess_users = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
    beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)
    gains = expected_marginal_gain_mixture_vec(
        tau_sessions,
        mu_matrix[sess_users],
        sigmas_by_user[sess_users],
        beliefs,
    )
    n_users = sigmas_by_user.shape[0]
    sum_gain = np.bincount(sess_users, weights=gains, minlength=n_users)
    n_sess = np.bincount(sess_users, minlength=n_users)
    observed_users = n_sess > 0
    c_new = np.full(n_users, float(c_floor), dtype=np.float64)
    c_new[observed_users] = sum_gain[observed_users] / n_sess[observed_users]
    c_new = np.maximum(c_new, float(c_floor))

    if float(log_shrinkage) > 0.0:
        eta = float(np.clip(log_shrinkage, 0.0, 1.0))
        log_c = np.log(c_new)
        center_source = log_c[observed_users] if observed_users.any() else log_c
        if shrink_target == "median":
            center = float(np.median(center_source))
        elif shrink_target == "mean":
            center = float(np.mean(center_source))
        else:
            raise ValueError("shrink_target must be 'mean' or 'median'")
        log_c[observed_users] = (1.0 - eta) * log_c[observed_users] + eta * center
        log_c[~observed_users] = center
        c_new = np.exp(log_c)

    return np.maximum(c_new, float(c_floor))


def update_theta_lbfgs_low_rank(
    interaction_df,
    session_table,
    responsibilities,
    tau_sessions_fixed,
    theta_old,
    sigmas_by_user,
    max_iter=25,
    reg_l2=THETA_L2_REG,
    score_anchor=None,
    score_anchor_weight=None,
    score_anchor_l2=SCORE_ANCHOR_L2,
    reference_cat_idx=None,
    c_log_shrinkage=SEARCH_COST_LOG_SHRINKAGE,
    c_shrink_target=SEARCH_COST_SHRINK_TARGET,
    max_obs=None,
    seed=0,
):
    """M-step for low-rank theta using L-BFGS on weighted probit likelihood."""
    if reference_cat_idx is None:
        reference_cat_idx = theta_old.get("reference_cat_idx", globals().get("UTILITY_REFERENCE_CAT_IDX", None))
    if reference_cat_idx is None:
        raise ValueError("reference_cat_idx is required for utility-location normalization")
    reference_cat_idx = int(reference_cat_idx)

    if score_anchor is None:
        score_anchor = globals().get("score_anchor_matrix", None)
    if score_anchor_weight is None:
        score_anchor_weight = globals().get("score_anchor_weight", None)

    rng = np.random.default_rng(seed)
    n_obs = len(interaction_df)
    if max_obs is not None and int(max_obs) < n_obs:
        obs_idx = np.sort(rng.choice(n_obs, size=int(max_obs), replace=False))
    else:
        obs_idx = np.arange(n_obs)

    u_idx = interaction_df["user_idx_int"].to_numpy(dtype=np.int64, copy=False)[obs_idx]
    k_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)[obs_idx]
    sess_idx = interaction_df["sess_idx"].to_numpy(dtype=np.int64, copy=False)[obs_idx]
    r = np.asarray(responsibilities, dtype=np.float64).reshape(-1)[obs_idx]
    tau_i = np.asarray(tau_sessions_fixed, dtype=np.float64).reshape(-1)[sess_idx]
    sigma_i = np.maximum(sigmas_by_user[u_idx, k_idx], 1e-8)

    device = torch.device("cpu")
    dtype = torch.float64
    u_t = torch.tensor(u_idx, dtype=torch.long, device=device)
    k_t = torch.tensor(k_idx, dtype=torch.long, device=device)
    r_t = torch.tensor(r, dtype=dtype, device=device)
    tau_t = torch.tensor(tau_i, dtype=dtype, device=device)
    sigma_t = torch.tensor(sigma_i, dtype=dtype, device=device)

    b_cat = torch.tensor(theta_old["b_cat"], dtype=dtype, device=device, requires_grad=True)
    U = torch.tensor(theta_old["U"], dtype=dtype, device=device, requires_grad=True)
    V = torch.tensor(theta_old["V"], dtype=dtype, device=device, requires_grad=True)
    params = [b_cat, U, V]

    use_score_anchor = (
        score_anchor is not None
        and score_anchor_weight is not None
        and float(score_anchor_l2) > 0.0
    )
    if use_score_anchor:
        score_target_t = torch.tensor(np.asarray(score_anchor, dtype=np.float64), dtype=dtype, device=device)
        score_weight_t = torch.tensor(np.asarray(score_anchor_weight, dtype=np.float64), dtype=dtype, device=device)
        if score_target_t.shape != (U.shape[0], b_cat.shape[0]):
            raise ValueError("score_anchor must have shape [N_users, K_cat]")
        score_weight_sum_t = torch.clamp(torch.sum(score_weight_t), min=1.0)

    optimizer = torch.optim.LBFGS(
        params,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
        tolerance_grad=1e-5,
        tolerance_change=1e-7,
    )

    sqrt2 = np.sqrt(2.0)
    eps = 1e-12

    def centered_mu_for_rows(row_users, row_cats):
        raw_mu = b_cat[row_cats] + torch.sum(U[row_users] * V[row_cats], dim=1)
        ref_mu = b_cat[reference_cat_idx] + torch.sum(U[row_users] * V[reference_cat_idx], dim=1)
        return raw_mu - ref_mu

    def centered_mu_matrix_torch():
        raw = b_cat[None, :] + U @ V.T
        return raw - raw[:, [reference_cat_idx]]

    def closure():
        optimizer.zero_grad()
        mu = centered_mu_for_rows(u_t, k_t)
        z = torch.clamp((mu - tau_t) / sigma_t, -10.0, 10.0)
        pi = torch.clamp(0.5 * (1.0 + torch.erf(z / sqrt2)), eps, 1.0 - eps)
        q = r_t * torch.log(pi) + (1.0 - r_t) * torch.log(1.0 - pi)
        neg_q = -torch.mean(q)

        reg = 0.5 * float(reg_l2) * (
            torch.mean(b_cat * b_cat)
            + torch.mean(U * U)
            + torch.mean(V * V)
        )

        if use_score_anchor:
            mu_full = centered_mu_matrix_torch()
            score_resid = mu_full - score_target_t
            score_loss = torch.sum(score_weight_t * score_resid * score_resid) / score_weight_sum_t
            reg = reg + 0.5 * float(score_anchor_l2) * score_loss

        loss = neg_q + reg
        loss.backward()
        return loss

    loss_before = float(closure().detach().cpu())
    optimizer.step(closure)
    loss_after = float(closure().detach().cpu())

    theta_new = {
        "b_cat": b_cat.detach().cpu().numpy(),
        "U": U.detach().cpu().numpy(),
        "V": V.detach().cpu().numpy(),
        "reference_cat_idx": reference_cat_idx,
    }
    theta_new["c"] = update_search_costs_from_thresholds(
        session_table=session_table,
        theta_without_c=theta_new,
        sigmas_by_user=sigmas_by_user,
        tau_sessions=tau_sessions_fixed,
        c_floor=COST_FLOOR,
        log_shrinkage=c_log_shrinkage,
        shrink_target=c_shrink_target,
    )

    score_anchor_rmse = np.nan
    if use_score_anchor:
        mu_np = theta_to_mu_matrix(theta_new, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
        w_np = np.asarray(score_anchor_weight, dtype=np.float64)
        target_np = np.asarray(score_anchor, dtype=np.float64)
        denom = max(float(w_np.sum()), 1.0)
        score_anchor_rmse = float(np.sqrt(np.sum(w_np * (mu_np - target_np) ** 2) / denom))

    info = {
        "theta_lbfgs_loss_before": loss_before,
        "theta_lbfgs_loss_after": loss_after,
        "theta_lbfgs_n_obs": int(len(obs_idx)),
        "theta_l2_reg": float(reg_l2),
        "score_anchor_l2": float(score_anchor_l2),
        "score_anchor_rmse": score_anchor_rmse,
        "c_log_shrinkage": float(c_log_shrinkage),
        "c_p01": float(np.percentile(theta_new["c"], 1)),
        "c_p50": float(np.percentile(theta_new["c"], 50)),
        "c_p99": float(np.percentile(theta_new["c"], 99)),
        "reference_mu_absmax": float(np.abs(theta_to_mu_matrix(theta_new)[:, reference_cat_idx]).max()),
    }
    return theta_new, info


def run_em_low_rank(
    interaction_df,
    session_table,
    theta_init,
    phi_init,
    sigmas_by_user,
    n_em_iters=10,
    tol=1e-4,
    gem_tol=GEM_TOL,
    phi_kwargs=None,
    theta_kwargs=None,
    orient_phi=True,
    phi_orientation_kwargs=None,
    verbose=True,
    keep_best_by_ll=True,
):
    phi_kwargs = {} if phi_kwargs is None else dict(phi_kwargs)
    theta_kwargs = {} if theta_kwargs is None else dict(theta_kwargs)
    phi_orientation_kwargs = {} if phi_orientation_kwargs is None else dict(phi_orientation_kwargs)
    def copy_param_dict(params):
        return {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in params.items()}

    theta = copy_param_dict(theta_init)
    theta.setdefault("reference_cat_idx", globals().get("UTILITY_REFERENCE_CAT_IDX"))
    theta["c"] = theta_to_cost_vector(theta, sigmas_by_user.shape[0])
    phi = copy_param_dict(phi_init)

    if orient_phi:
        phi, _ = orient_phi_high_watch_state(interaction_df, phi, **phi_orientation_kwargs)

    history = []
    prev_ll = -np.inf
    best_ll = -np.inf
    best_iter = 0
    best_stage = "initial"
    best_theta = copy_param_dict(theta)
    best_phi = copy_param_dict(phi)

    for em_iter in range(1, int(n_em_iters) + 1):
        tic = time()
        best_stage_this_iter = "none"
        r_i, tau_i, ll_before, tau_sessions = compute_e_step_with_ll(
            interaction_df,
            session_table,
            theta,
            phi,
            sigmas_by_user,
            return_session_tau=True,
            diagnostic_label=f"iter{em_iter}_before_mstep",
        )
        ll_gain_from_prev = ll_before - prev_ll
        if keep_best_by_ll and float(ll_before) > best_ll:
            best_ll = float(ll_before)
            best_iter = em_iter - 1
            best_stage = "before_m_step"
            best_theta = copy_param_dict(theta)
            best_phi = copy_param_dict(phi)
            best_stage_this_iter = "before_m_step"

        phi_candidate = update_phi_coordinate_descent(
            interaction_df=interaction_df,
            responsibilities=r_i,
            phi_old=phi,
            **phi_kwargs,
        )
        phi_orient_info = {}
        if orient_phi:
            phi_candidate, phi_orient_info = orient_phi_high_watch_state(
                interaction_df,
                phi_candidate,
                **phi_orientation_kwargs,
            )

        r_i, tau_i, ll_after_phi, tau_sessions = compute_e_step_with_ll(
            interaction_df,
            session_table,
            theta,
            phi_candidate,
            sigmas_by_user,
            return_session_tau=True,
            diagnostic_label=f"iter{em_iter}_after_phi",
        )
        if keep_best_by_ll and float(ll_after_phi) > best_ll:
            best_ll = float(ll_after_phi)
            best_iter = em_iter
            best_stage = "after_phi"
            best_theta = copy_param_dict(theta)
            best_phi = copy_param_dict(phi_candidate)
            best_stage_this_iter = "after_phi"

        theta_before_theta = copy_param_dict(theta)
        phi = phi_candidate
        theta_candidate, theta_info = update_theta_lbfgs_low_rank(
            interaction_df=interaction_df,
            session_table=session_table,
            responsibilities=r_i,
            tau_sessions_fixed=tau_sessions,
            theta_old=theta_before_theta,
            sigmas_by_user=sigmas_by_user,
            **theta_kwargs,
        )

        _, _, ll_after_theta_candidate = compute_e_step_with_ll(
            interaction_df,
            session_table,
            theta_candidate,
            phi,
            sigmas_by_user,
            diagnostic_label=f"iter{em_iter}_after_theta_candidate",
        )
        theta_update_rejected = bool(ll_after_theta_candidate < ll_after_phi - float(gem_tol))
        theta_info = dict(theta_info) if theta_info is not None else {}
        theta_info["theta_update_rejected"] = bool(theta_update_rejected)
        if theta_update_rejected:
            theta = theta_before_theta
            ll_after_theta = float(ll_after_phi)
            theta_info["theta_info_refers_to_rejected_candidate"] = True
            theta_info["accepted_theta_ll"] = float(ll_after_phi)
            theta_info["rejected_theta_candidate_ll"] = float(ll_after_theta_candidate)
        else:
            theta = theta_candidate
            ll_after_theta = float(ll_after_theta_candidate)
            theta_info["theta_info_refers_to_rejected_candidate"] = False
            theta_info["accepted_theta_ll"] = float(ll_after_theta_candidate)
            theta_info["rejected_theta_candidate_ll"] = np.nan

        theta_ll_decreased = bool(ll_after_theta < ll_after_phi - 1e-6)
        em_ll_decreased = bool(ll_after_theta < ll_before - 1e-6)
        best_updated = False
        if keep_best_by_ll and (not theta_update_rejected) and float(ll_after_theta) > best_ll:
            best_ll = float(ll_after_theta)
            best_iter = em_iter
            best_stage = "after_theta"
            best_theta = copy_param_dict(theta)
            best_phi = copy_param_dict(phi)
            best_stage_this_iter = "after_theta"
            best_updated = True

        record = {
            "iter": em_iter,
            "ll_before": float(ll_before),
            "ll_after_phi": float(ll_after_phi),
            "ll_after_theta_candidate": float(ll_after_theta_candidate),
            "ll_after": float(ll_after_theta),
            "ll_delta_after_phi_step": float(ll_after_phi - ll_before),
            "ll_delta_after_theta_step": float(ll_after_theta - ll_after_phi),
            "ll_delta_after_mstep": float(ll_after_theta - ll_before),
            "ll_gain_from_prev": float(ll_gain_from_prev),
            "theta_update_rejected": theta_update_rejected,
            "theta_ll_decreased": theta_ll_decreased,
            "em_ll_decreased": em_ll_decreased,
            "best_ll_so_far": float(best_ll),
            "best_iter": int(best_iter),
            "best_stage_so_far": best_stage,
            "best_stage_this_iter": best_stage_this_iter,
            "is_best_iter": bool(best_updated),
            "seconds": float(time() - tic),
            **phi_orient_info,
            **theta_info,
        }
        history.append(record)
        if verbose:
            print(record)

        if em_iter > 1 and abs(ll_after_theta - prev_ll) < float(tol):
            break
        prev_ll = ll_after_theta

    if keep_best_by_ll:
        last_iter = int(history[-1]["iter"]) if history else 0
        if verbose and (best_iter != last_iter or best_stage != "after_theta"):
            print(f"Returning best observed-LL snapshot from iter {best_iter}, stage {best_stage} (best_ll={best_ll:.6f})")
        return best_theta, best_phi, pd.DataFrame(history)

    return theta, phi, pd.DataFrame(history)


In [ ]:
def make_em_session_subset(interaction_df, session_table, n_sessions=500, seed=0):
    """Create a small internally reindexed session/interactions subset for debugging EM."""
    rng = np.random.default_rng(seed)
    n_sessions = int(min(n_sessions, len(session_table)))
    chosen_old_sess = np.sort(rng.choice(session_table["sess_idx"].to_numpy(), size=n_sessions, replace=False))

    sub_sessions = (
        session_table[session_table["sess_idx"].isin(chosen_old_sess)]
        .sort_values(["user_idx", "burst_id", "session"])
        .copy()
        .reset_index(drop=True)
    )
    old_to_new = {old: new for new, old in enumerate(sub_sessions["sess_idx"].to_numpy())}
    sub_sessions["sess_idx"] = np.arange(len(sub_sessions), dtype=np.int64)

    sub_interactions = interaction_df[interaction_df["sess_idx"].isin(chosen_old_sess)].copy()
    sub_interactions["sess_idx"] = sub_interactions["sess_idx"].map(old_to_new).astype(np.int64)
    return sub_interactions.reset_index(drop=True), sub_sessions


RUN_EM_SMOKE_TEST = False

if RUN_EM_SMOKE_TEST:
    smoke_df, smoke_sessions = make_em_session_subset(interaction_df, session_table, n_sessions=300, seed=1)
    smoke_theta = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in theta_init.items()}
    smoke_phi = init_phi_from_interactions(smoke_df, cat_ids=cat_ids, seed=1)
    theta_smoke_hat, phi_smoke_hat, hist_smoke = run_em_low_rank(
        interaction_df=smoke_df,
        session_table=smoke_sessions,
        theta_init=smoke_theta,
        phi_init=smoke_phi,
        sigmas_by_user=sigmas_by_user,
        n_em_iters=2,
        phi_kwargs={"n_coord_iters": 1, "logvar_maxiter": 5},
        theta_kwargs={
            "max_iter": 5,
            "max_obs": 50_000,
            "reg_l2": THETA_L2_REG,
            "c_log_shrinkage": SEARCH_COST_LOG_SHRINKAGE,
            "c_shrink_target": SEARCH_COST_SHRINK_TARGET,
            "score_anchor_l2": SCORE_ANCHOR_L2,
            "seed": 1,
        },
        orient_phi=True,
        phi_orientation_kwargs={"sample_size": 100_000, "seed": 1},
        verbose=True,
    )
    display(hist_smoke)

## 6. Session-Level Train/Validation Likelihood Diagnostic

This optional diagnostic estimates the model on a session subset and evaluates observed likelihood on held-out sessions. It is off by default because the full EM run is already expensive.


In [ ]:
def make_session_train_val_split(
    session_table,
    val_frac=0.20,
    method="last",
    min_train_sessions=1,
    seed=0,
):
    """Return old sess_idx arrays for a within-user session train/validation split."""
    st = (
        session_table[["sess_idx", "user_idx", "user_id", "burst_id", "session"]]
        .sort_values(["user_idx", "burst_id", "session"])
        .reset_index(drop=True)
        .copy()
    )
    st["session_order"] = st.groupby("user_idx").cumcount()
    st["n_sessions_user"] = st.groupby("user_idx")["sess_idx"].transform("count")

    if method == "last":
        n_val = np.floor(st["n_sessions_user"].to_numpy(dtype=float) * float(val_frac)).astype(int)
        eligible = st["n_sessions_user"].to_numpy(dtype=int) > int(min_train_sessions)
        n_val = np.where(eligible & (n_val < 1), 1, n_val)
        n_val = np.minimum(n_val, st["n_sessions_user"].to_numpy(dtype=int) - int(min_train_sessions))
        n_val = np.maximum(n_val, 0)
        st["is_val"] = eligible & (st["session_order"].to_numpy(dtype=int) >= st["n_sessions_user"].to_numpy(dtype=int) - n_val)
    elif method == "random":
        rng = np.random.default_rng(seed)
        is_val = np.zeros(len(st), dtype=bool)
        for _, group in st.groupby("user_idx", sort=False):
            n_sess = len(group)
            if n_sess <= int(min_train_sessions):
                continue
            n_val = int(np.floor(n_sess * float(val_frac)))
            if n_val < 1:
                n_val = 1
            n_val = min(n_val, n_sess - int(min_train_sessions))
            chosen_pos = rng.choice(group.index.to_numpy(), size=n_val, replace=False)
            is_val[chosen_pos] = True
        st["is_val"] = is_val
    else:
        raise ValueError("method must be 'last' or 'random'")

    val_old_sess = st.loc[st["is_val"], "sess_idx"].to_numpy(dtype=np.int64)
    train_old_sess = st.loc[~st["is_val"], "sess_idx"].to_numpy(dtype=np.int64)
    return train_old_sess, val_old_sess, st


def make_reindexed_session_subset(interaction_df, session_table, old_sess_idx):
    """Subset interactions/sessions by old sess_idx and reindex sess_idx to 0..S-1."""
    old_sess_idx = np.asarray(sorted(set(np.asarray(old_sess_idx, dtype=np.int64).tolist())), dtype=np.int64)
    sub_sessions = (
        session_table[session_table["sess_idx"].isin(old_sess_idx)]
        .sort_values(["user_idx", "burst_id", "session"])
        .copy()
        .reset_index(drop=True)
    )
    sub_sessions["old_sess_idx"] = sub_sessions["sess_idx"].astype(np.int64)
    old_to_new = {old: new for new, old in enumerate(sub_sessions["old_sess_idx"].to_numpy(dtype=np.int64))}
    sub_sessions["sess_idx"] = np.arange(len(sub_sessions), dtype=np.int64)

    sub_interactions = interaction_df[interaction_df["sess_idx"].isin(old_sess_idx)].copy()
    sub_interactions["old_sess_idx"] = sub_interactions["sess_idx"].astype(np.int64)
    sub_interactions["sess_idx"] = sub_interactions["old_sess_idx"].map(old_to_new).astype(np.int64)
    return sub_interactions.reset_index(drop=True), sub_sessions


def compute_score_moments_for_sessions(df_valid, subset_session_table):
    """Recompute normalized score moments using only rows from a session subset."""
    keys = subset_session_table[["user_id", "burst_id", "session"]].drop_duplicates()
    df_score = df_valid.merge(keys, on=["user_id", "burst_id", "session"], how="inner").copy()
    if df_score.empty:
        raise ValueError("No rows found for the requested session subset")

    raw_score = np.zeros(len(df_score), dtype=np.float64)
    for col, weight in SCORE_WEIGHTS.items():
        if col not in df_score.columns:
            raise KeyError(f"df_valid is missing score column {col}")
        raw_score += float(weight) * pd.to_numeric(df_score[col], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
    raw_score *= float(SCORE_SCALE)
    df_score["raw_score_for_subset"] = raw_score

    ref_mask = df_score["i_cat_level1_id"].astype(int).eq(int(SCORE_REFERENCE_CATEGORY_ID))
    ref_by_user = df_score.loc[ref_mask].groupby("user_id")["raw_score_for_subset"].mean()
    global_ref = float(df_score.loc[ref_mask, "raw_score_for_subset"].mean()) if ref_mask.any() else 0.0
    if not np.isfinite(global_ref):
        global_ref = 0.0
    df_score["score_for_subset"] = (df_score["raw_score_for_subset"] - df_score["user_id"].map(shift_map).fillna(global_ref)) / df_score["user_id"].map(scale_map).fillna(global_reliable_raw_sigma)

    return (
        df_score
        .groupby(["user_id", "i_cat_level1_id"], observed=True)["score_for_subset"]
        .agg(n_score="size", mean_score="mean", var_score=lambda x: x.var(ddof=0))
        .reset_index()
    )


def safe_corr(x, y):
    """Correlation that returns NaN instead of failing on constant or empty inputs."""
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]
    y = y[mask]
    if np.std(x) == 0.0 or np.std(y) == 0.0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def observed_loglik_summary(interaction_df, session_table, theta, phi, sigmas_by_user, split_name="sample"):
    """Evaluate observed likelihood and return per-observation and per-session summaries."""
    if len(interaction_df) == 0:
        raise ValueError("Cannot evaluate likelihood on an empty interaction_df")

    mu_matrix = theta_to_mu_matrix(theta, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
    cost_vector = theta_to_cost_vector(theta, n_users=sigmas_by_user.shape[0])
    sess_users = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
    beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)
    tau_sessions = solve_tau_vectorized(
        mu_matrix[sess_users],
        sigmas_by_user[sess_users],
        beliefs,
        cost_vector[sess_users],
        diagnostic_label=f"observed_loglik_{split_name}",
    )

    u_idx = interaction_df["user_idx_int"].to_numpy(dtype=np.int64, copy=False)
    k_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)
    s_idx = interaction_df["sess_idx"].to_numpy(dtype=np.int64, copy=False)
    tau_i = tau_sessions[s_idx]
    sigma_i = np.maximum(sigmas_by_user[u_idx, k_idx], 1e-8)
    mu_i = mu_matrix[u_idx, k_idx]
    pi_i = np.clip(standard_normal_cdf((mu_i - tau_i) / sigma_i), 1e-12, 1.0 - 1e-12)

    y = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
    m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, return_logvar=True)
    log_joint1 = np.log(pi_i) + log_normal_pdf_from_logvar(y, m1, logv1)
    log_joint0 = np.log1p(-pi_i) + log_normal_pdf_from_logvar(y, m0, logv0)
    row_ll = np.logaddexp(log_joint1, log_joint0)
    r_hat = np.exp(log_joint1 - row_ll)
    if "watch_ratio" in interaction_df.columns:
        watch_ratio = pd.to_numeric(interaction_df["watch_ratio"], errors="coerce").to_numpy(dtype=np.float64)
    else:
        watch_ratio = np.exp(y)

    total_ll = float(np.sum(row_ll))
    session_ll = np.bincount(s_idx, weights=row_ll, minlength=len(session_table))
    session_n = np.bincount(s_idx, minlength=len(session_table))
    observed_sessions = session_n > 0
    session_avg_ll = session_ll[observed_sessions] / session_n[observed_sessions]

    return {
        "split": split_name,
        "n_obs": int(len(interaction_df)),
        "n_sessions": int(observed_sessions.sum()),
        "total_ll": total_ll,
        "avg_ll_per_obs": total_ll / float(len(interaction_df)),
        "avg_session_avg_ll": float(np.mean(session_avg_ll)),
        "median_session_avg_ll": float(np.median(session_avg_ll)),
        "mean_r_hat": float(np.mean(r_hat)),
        "corr_r_watch_ratio": safe_corr(r_hat, watch_ratio),
        "corr_r_log_watch_ratio": safe_corr(r_hat, y),
    }


RUN_SESSION_VALIDATION_DIAGNOSTIC = False

if RUN_SESSION_VALIDATION_DIAGNOSTIC:
    VAL_FRAC = 0.20
    VAL_SPLIT_METHOD = "last"  # hold out the last sessions within each user
    TRAIN_VAL_EM_ITERS = 30
    TRAIN_VAL_TOL = 50.0

    train_old_sess, val_old_sess, split_session_table = make_session_train_val_split(
        session_table,
        val_frac=VAL_FRAC,
        method=VAL_SPLIT_METHOD,
        min_train_sessions=1,
        seed=2031,
    )
    train_interaction_df, train_session_table = make_reindexed_session_subset(interaction_df, session_table, train_old_sess)
    val_interaction_df, val_session_table = make_reindexed_session_subset(interaction_df, session_table, val_old_sess)

    train_score_moments = compute_score_moments_for_sessions(df_valid, train_session_table)
    theta_init_train = init_theta_low_rank_from_scores(
        user_cat_score_moments=train_score_moments,
        user_ids=user_ids,
        cat_ids=cat_ids,
        d=LATENT_DIM,
        initial_search_cost=INITIAL_SEARCH_COST,
    )
    theta_init_train["reference_cat_idx"] = UTILITY_REFERENCE_CAT_IDX
    theta_init_train["c"] = initial_cost_by_user.copy()
    score_anchor_train, score_anchor_weight_train = build_score_anchor_inputs(
        user_cat_score_moments=train_score_moments,
        user_ids=user_ids,
        cat_ids=cat_ids,
        min_n=SCORE_ANCHOR_MIN_N,
        n0=SCORE_ANCHOR_N0,
        sparse_min_weight=SPARSE_SCORE_ANCHOR_MIN_WEIGHT,
        unobserved_weight=UNOBSERVED_SCORE_ANCHOR_WEIGHT,
    )
    phi_init_train = init_phi_from_interactions(train_interaction_df, cat_ids=cat_ids, seed=2031)

    theta_tv, phi_tv, hist_tv = run_em_low_rank(
        interaction_df=train_interaction_df,
        session_table=train_session_table,
        theta_init=theta_init_train,
        phi_init=phi_init_train,
        sigmas_by_user=sigmas_by_user,
        n_em_iters=TRAIN_VAL_EM_ITERS,
        tol=TRAIN_VAL_TOL,
        phi_kwargs={
            "n_coord_iters": 1,
            "ridge_mean": 1e-6,
            "ridge_logvar": 1e-6,
            "logvar_maxiter": 10,
        },
        theta_kwargs={
            "max_iter": 50,
            "max_obs": None,
            "reg_l2": THETA_L2_REG,
            "score_anchor_l2": SCORE_ANCHOR_L2,
            "score_anchor": score_anchor_train,
            "score_anchor_weight": score_anchor_weight_train,
            "reference_cat_idx": UTILITY_REFERENCE_CAT_IDX,
            "c_log_shrinkage": SEARCH_COST_LOG_SHRINKAGE,
            "c_shrink_target": SEARCH_COST_SHRINK_TARGET,
            "seed": 2031,
        },
        orient_phi=True,
        phi_orientation_kwargs={"sample_size": 200_000, "seed": 2031},
        verbose=True,
    )

    train_ll_summary = observed_loglik_summary(
        train_interaction_df, train_session_table, theta_tv, phi_tv, sigmas_by_user, split_name="train"
    )
    val_ll_summary = observed_loglik_summary(
        val_interaction_df, val_session_table, theta_tv, phi_tv, sigmas_by_user, split_name="validation"
    )
    train_val_ll_summary = pd.DataFrame([train_ll_summary, val_ll_summary])
    train_val_gap_summary = pd.DataFrame([{
        "avg_ll_per_obs_train": train_ll_summary["avg_ll_per_obs"],
        "avg_ll_per_obs_validation": val_ll_summary["avg_ll_per_obs"],
        "avg_ll_per_obs_gap_train_minus_val": train_ll_summary["avg_ll_per_obs"] - val_ll_summary["avg_ll_per_obs"],
        "avg_session_avg_ll_train": train_ll_summary["avg_session_avg_ll"],
        "avg_session_avg_ll_validation": val_ll_summary["avg_session_avg_ll"],
        "avg_session_avg_ll_gap_train_minus_val": train_ll_summary["avg_session_avg_ll"] - val_ll_summary["avg_session_avg_ll"],
        "corr_r_watch_ratio_train": train_ll_summary["corr_r_watch_ratio"],
        "corr_r_watch_ratio_validation": val_ll_summary["corr_r_watch_ratio"],
        "corr_r_log_watch_ratio_train": train_ll_summary["corr_r_log_watch_ratio"],
        "corr_r_log_watch_ratio_validation": val_ll_summary["corr_r_log_watch_ratio"],
    }])

    comparison_cols = [
        "split", "n_obs", "n_sessions", "total_ll", "avg_ll_per_obs",
        "avg_session_avg_ll", "median_session_avg_ll", "mean_r_hat",
        "corr_r_watch_ratio", "corr_r_log_watch_ratio",
    ]
    display(train_val_ll_summary[comparison_cols])
    display(train_val_gap_summary)
    display(hist_tv)


## 7. Full-Data EM Estimation and Diagnostics

The full run below is configured to use all observations. For quick checking, run the smoke-test cell first by setting `RUN_EM_SMOKE_TEST = True`.

Saved diagnostics include:

| File | Meaning |
|---|---|
| `em_history.parquet` | Iteration-level EM likelihood and acceptance history |
| `threshold_residual_diagnostics.csv` | Bisection residual summaries from threshold solves |
| `logvar_optimizer_diagnostics.csv` | L-BFGS diagnostics for log-variance blocks |
| `search_cost_summary.csv` | Final search cost percentiles and user-level costs |
| `utility_scale_normalization_summary.csv` | Per-user shift/scale source for UNKNOWN normalization |
| `structural_interactions.parquet` | Reusable interaction rows/features for counterfactual notebooks |
| `structural_sessions.parquet` | Reusable session rows plus baseline cost/threshold |
| `structural_estimation_arrays.npz` | `sigmas_by_user`, `session_beliefs`, ids, and compact arrays |
| `structural_baseline_posteriors.parquet` | Baseline final posterior responsibility `r_hat` for each interaction |


In [ ]:

# Corrected full-data EM estimation.
# Uses all observations, d=15 default, score anchor, unobserved neutral anchor,
# per-user UNKNOWN N(0,1) normalization, and z=1 high-watch orientation.

FULL_EM_ITERS = 200
FULL_EM_TOL = 50.0

threshold_diagnostic_records.clear()
logvar_optimizer_diagnostic_records.clear()

theta_corr, phi_corr, hist_corr = run_em_low_rank(
    interaction_df=interaction_df,
    session_table=session_table,
    theta_init={k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in theta_init.items()},
    phi_init={k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in phi_init.items()},
    sigmas_by_user=sigmas_by_user,
    n_em_iters=FULL_EM_ITERS,
    tol=FULL_EM_TOL,
    gem_tol=GEM_TOL,
    phi_kwargs={
        "n_coord_iters": 1,
        "ridge_mean": 1e-6,
        "ridge_logvar": 1e-6,
        "logvar_maxiter": 10,
    },
    theta_kwargs={
        "max_iter": 50,
        "max_obs": None,
        "reg_l2": THETA_L2_REG,
        "score_anchor_l2": SCORE_ANCHOR_L2,
        "score_anchor": score_anchor_matrix,
        "score_anchor_weight": score_anchor_weight,
        "reference_cat_idx": UTILITY_REFERENCE_CAT_IDX,
        "c_log_shrinkage": SEARCH_COST_LOG_SHRINKAGE,
        "c_shrink_target": SEARCH_COST_SHRINK_TARGET,
        "seed": 2030,
    },
    orient_phi=True,
    phi_orientation_kwargs={
        "sample_size": 200_000,
        "seed": 2030,
    },
    verbose=True,
)

hist_corr = hist_corr.copy()
hist_corr["neg_ll_before"] = -hist_corr["ll_before"]
hist_corr["neg_ll_after"] = -hist_corr["ll_after"]
hist_corr["relative_ll_gain"] = hist_corr["ll_delta_after_mstep"] / hist_corr["ll_before"].abs()

display_cols = [
    "iter",
    "ll_before",
    "ll_after_phi",
    "ll_after_theta_candidate",
    "ll_after",
    "ll_delta_after_phi_step",
    "ll_delta_after_theta_step",
    "ll_delta_after_mstep",
    "relative_ll_gain",
    "theta_update_rejected",
    "theta_info_refers_to_rejected_candidate",
    "accepted_theta_ll",
    "rejected_theta_candidate_ll",
    "theta_ll_decreased",
    "em_ll_decreased",
    "best_iter",
    "best_stage_so_far",
    "best_stage_this_iter",
    "seconds",
    "phi_state_swapped",
    "phi_mean_m0",
    "phi_mean_m1",
    "phi_mean_m1_minus_m0",
    "theta_lbfgs_loss_before",
    "theta_lbfgs_loss_after",
    "score_anchor_l2",
    "score_anchor_rmse",
    "reference_mu_absmax",
    "c_p01",
    "c_p50",
    "c_p99",
]
display(hist_corr[[col for col in display_cols if col in hist_corr.columns]])


def ensure_row_id_in_interactions(frame):
    """Attach original processed-data row_id if this kernel was started before row_id was added.

    Notebook 02 uses the processed-data row order as GNN edge row order. New runs keep
    row_id from the start; this fallback lets an already-finished notebook-06 kernel
    save the bundle without rerunning EM.
    """
    if "row_id" in frame.columns:
        out = frame.copy()
        out["row_id"] = out["row_id"].astype(np.int64)
        return out

    candidate_cols = [
        "user_id", "video_id", "burst_id", "session", "date", "time", "timestamp",
        "i_cat_level1_id", "i_video_duration", "sess_rank",
    ]
    merge_cols = [col for col in candidate_cols if col in frame.columns]
    if len(merge_cols) < 6:
        raise RuntimeError(
            "Cannot reconstruct row_id because interaction_df has too few stable key columns. "
            "Rerun notebook 06 from the data-loading cells so row_id is preserved."
        )

    source = pd.read_parquet(DATA_PATH, columns=merge_cols)
    source = source.reset_index(names="row_id")

    # Match dtypes on both sides before merge.
    left = frame.copy()
    for col in merge_cols:
        if col in ["user_id", "video_id", "burst_id", "session", "i_cat_level1_id"]:
            left[col] = pd.to_numeric(left[col], errors="coerce").astype(source[col].dtype)
        else:
            left[col] = left[col].astype(source[col].dtype, copy=False)

    if source.duplicated(merge_cols).any():
        dup = int(source.duplicated(merge_cols).sum())
        raise RuntimeError(
            f"Cannot reconstruct row_id safely: source keys have {dup} duplicate rows. "
            "Rerun notebook 06 from the data-loading cells so row_id is preserved exactly."
        )

    out = left.merge(source[merge_cols + ["row_id"]], on=merge_cols, how="left", validate="many_to_one")
    missing = int(out["row_id"].isna().sum())
    if missing:
        raise RuntimeError(f"Failed to reconstruct row_id for {missing} interaction rows")
    out["row_id"] = out["row_id"].astype(np.int64)
    return out

# Save core diagnostics immediately after the run.
hist_corr.to_parquet(OUTPUT_DIR / "em_history.parquet", index=False)

logvar_diag_df = pd.DataFrame(logvar_optimizer_diagnostic_records)
logvar_diag_df.to_csv(OUTPUT_DIR / "logvar_optimizer_diagnostics.csv", index=False)

cost_df = pd.DataFrame({
    "user_id": user_ids,
    "user_idx": np.arange(len(user_ids)),
    "c": theta_corr["c"],
    "log_c": np.log(theta_to_cost_vector(theta_corr, N_users)),
})
cost_df.to_csv(OUTPUT_DIR / "search_cost_summary.csv", index=False)

utility_norm_summary.to_csv(OUTPUT_DIR / "utility_scale_normalization_summary.csv", index=False)

# Save reusable estimation-sample artifacts for downstream notebooks.
# Notebook 07 should load these instead of rebuilding users, sessions, beliefs, and sigmas.
final_r_hat, final_tau_i, final_ll, final_tau_sessions = compute_e_step_with_ll(
    interaction_df=interaction_df,
    session_table=session_table,
    theta=theta_corr,
    phi=phi_corr,
    sigmas_by_user=sigmas_by_user,
    return_session_tau=True,
    diagnostic_label="final_saved_baseline_posterior",
)

threshold_diag_df = pd.DataFrame(threshold_diagnostic_records)
threshold_diag_df.to_csv(OUTPUT_DIR / "threshold_residual_diagnostics.csv", index=False)

session_beliefs_matrix = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)
session_bundle = session_table.drop(columns=["belief_vec"]).copy()
session_bundle["cost_hat"] = theta_to_cost_vector(theta_corr, N_users)[session_bundle["user_idx"].to_numpy(dtype=np.int64)]
session_bundle["tau_hat"] = final_tau_sessions.astype(np.float64)
session_bundle["belief_entropy_hat"] = (-session_beliefs_matrix * np.log(np.maximum(session_beliefs_matrix, 1e-12))).sum(axis=1)

interaction_bundle = ensure_row_id_in_interactions(interaction_df)
interaction_bundle.to_parquet(OUTPUT_DIR / "structural_interactions.parquet", index=False)
session_bundle.to_parquet(OUTPUT_DIR / "structural_sessions.parquet", index=False)

baseline_posterior_df = interaction_bundle[[
    "row_id", "user_id", "video_id", "burst_id", "session", "sess_idx",
    "i_cat_level1_id", "user_idx_int", "cat_idx_int", "log_watch_ratio",
]].copy()
baseline_posterior_df["r_hat"] = final_r_hat.astype(np.float32)
baseline_posterior_df["tau_hat"] = final_tau_i.astype(np.float32)
baseline_posterior_df.to_parquet(OUTPUT_DIR / "structural_baseline_posteriors.parquet", index=False)

np.savez_compressed(
    OUTPUT_DIR / "structural_estimation_arrays.npz",
    sigmas_by_user=sigmas_by_user,
    session_beliefs=session_beliefs_matrix,
    obs_user_idx=obs_user_idx,
    obs_cat_idx=obs_cat_idx,
    obs_sess_idx=obs_sess_idx,
    obs_log_watch_ratio=obs_log_watch_ratio,
    user_ids=user_ids,
    cat_ids=cat_ids,
    reference_cat_idx=np.array([UTILITY_REFERENCE_CAT_IDX], dtype=np.int64),
)

bundle_metadata = {
    "n_users": int(N_users),
    "n_categories": int(K_cat),
    "n_interactions": int(len(interaction_df)),
    "n_sessions": int(len(session_table)),
    "burn_in_days": int(BURN_IN_DAYS),
    "min_obs_per_user": int(MIN_OBS_PER_USER),
    "score_weight_spec": HEAD_WEIGHT_SPEC,
    "score_weight_source": SCORE_WEIGHT_SOURCE,
    "score_weights": {key: float(value) for key, value in SCORE_WEIGHTS.items()},
    "baseline_observed_ll": float(final_ll),
    "row_id_alignment": "row_id equals the original processed_data row index and gnn_data edge row",
    "paths": {
        "interactions": str(OUTPUT_DIR / "structural_interactions.parquet"),
        "sessions": str(OUTPUT_DIR / "structural_sessions.parquet"),
        "arrays": str(OUTPUT_DIR / "structural_estimation_arrays.npz"),
        "baseline_posteriors": str(OUTPUT_DIR / "structural_baseline_posteriors.parquet"),
    },
}
(OUTPUT_DIR / "structural_estimation_bundle_metadata.json").write_text(json.dumps(bundle_metadata, indent=2))

np.savez_compressed(
    OUTPUT_DIR / "structural_estimates_theta_phi.npz",
    theta_b_cat=theta_corr["b_cat"],
    theta_U=theta_corr["U"],
    theta_V=theta_corr["V"],
    theta_c=theta_corr["c"],
    phi_alpha_mu0=phi_corr["alpha_mu0"],
    phi_alpha_mu1=phi_corr["alpha_mu1"],
    phi_beta_mu0=phi_corr["beta_mu0"],
    phi_beta_mu1=phi_corr["beta_mu1"],
    phi_alpha_logvar0=phi_corr["alpha_logvar0"],
    phi_alpha_logvar1=phi_corr["alpha_logvar1"],
    phi_beta_logvar0=phi_corr["beta_logvar0"],
    phi_beta_logvar1=phi_corr["beta_logvar1"],
    user_ids=user_ids,
    cat_ids=cat_ids,
    reference_cat_idx=np.array([UTILITY_REFERENCE_CAT_IDX], dtype=np.int64),
)

print("saved outputs to", OUTPUT_DIR)
print("saved structural bundle files:")
print(" -", OUTPUT_DIR / "structural_interactions.parquet")
print(" -", OUTPUT_DIR / "structural_sessions.parquet")
print(" -", OUTPUT_DIR / "structural_estimation_arrays.npz")
print(" -", OUTPUT_DIR / "structural_baseline_posteriors.parquet")
print("threshold diagnostics rows:", len(threshold_diag_df))
print("logvar optimizer diagnostics rows:", len(logvar_diag_df))


### Save-Only Bundle Cell

If the full EM cell above has already finished in the current kernel, run this cell to save/re-save the reusable structural bundle without rerunning EM. This is useful for notebook 07.


In [ ]:


def ensure_row_id_in_interactions(frame):
    """Attach original processed-data row_id if this kernel was started before row_id was added.

    Notebook 02 uses the processed-data row order as GNN edge row order. New runs keep
    row_id from the start; this fallback lets an already-finished notebook-06 kernel
    save the bundle without rerunning EM.
    """
    if "row_id" in frame.columns:
        out = frame.copy()
        out["row_id"] = out["row_id"].astype(np.int64)
        return out

    candidate_cols = [
        "user_id", "video_id", "burst_id", "session", "date", "time", "timestamp",
        "i_cat_level1_id", "i_video_duration", "sess_rank",
    ]
    merge_cols = [col for col in candidate_cols if col in frame.columns]
    if len(merge_cols) < 6:
        raise RuntimeError(
            "Cannot reconstruct row_id because interaction_df has too few stable key columns. "
            "Rerun notebook 06 from the data-loading cells so row_id is preserved."
        )

    source = pd.read_parquet(DATA_PATH, columns=merge_cols)
    source = source.reset_index(names="row_id")

    # Match dtypes on both sides before merge.
    left = frame.copy()
    for col in merge_cols:
        if col in ["user_id", "video_id", "burst_id", "session", "i_cat_level1_id"]:
            left[col] = pd.to_numeric(left[col], errors="coerce").astype(source[col].dtype)
        else:
            left[col] = left[col].astype(source[col].dtype, copy=False)

    if source.duplicated(merge_cols).any():
        dup = int(source.duplicated(merge_cols).sum())
        raise RuntimeError(
            f"Cannot reconstruct row_id safely: source keys have {dup} duplicate rows. "
            "Rerun notebook 06 from the data-loading cells so row_id is preserved exactly."
        )

    out = left.merge(source[merge_cols + ["row_id"]], on=merge_cols, how="left", validate="many_to_one")
    missing = int(out["row_id"].isna().sum())
    if missing:
        raise RuntimeError(f"Failed to reconstruct row_id for {missing} interaction rows")
    out["row_id"] = out["row_id"].astype(np.int64)
    return out

# Save/re-save reusable structural artifacts without rerunning EM.
required_existing_objects = [
    "theta_corr", "phi_corr", "hist_corr", "interaction_df", "session_table",
    "sigmas_by_user", "user_ids", "cat_ids", "UTILITY_REFERENCE_CAT_IDX",
]
missing_objects = [name for name in required_existing_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        "Run the full EM estimation cell first, or load these objects before using the save-only cell. "
        f"Missing: {missing_objects}"
    )

final_r_hat, final_tau_i, final_ll, final_tau_sessions = compute_e_step_with_ll(
    interaction_df=interaction_df,
    session_table=session_table,
    theta=theta_corr,
    phi=phi_corr,
    sigmas_by_user=sigmas_by_user,
    return_session_tau=True,
    diagnostic_label="final_saved_baseline_posterior_save_only",
)

session_beliefs_matrix = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)
session_bundle = session_table.drop(columns=["belief_vec"]).copy()
session_bundle["cost_hat"] = theta_to_cost_vector(theta_corr, len(user_ids))[session_bundle["user_idx"].to_numpy(dtype=np.int64)]
session_bundle["tau_hat"] = final_tau_sessions.astype(np.float64)
session_bundle["belief_entropy_hat"] = (-session_beliefs_matrix * np.log(np.maximum(session_beliefs_matrix, 1e-12))).sum(axis=1)

interaction_bundle = ensure_row_id_in_interactions(interaction_df)
interaction_bundle.to_parquet(OUTPUT_DIR / "structural_interactions.parquet", index=False)
session_bundle.to_parquet(OUTPUT_DIR / "structural_sessions.parquet", index=False)

baseline_posterior_df = interaction_bundle[[
    "row_id", "user_id", "video_id", "burst_id", "session", "sess_idx",
    "i_cat_level1_id", "user_idx_int", "cat_idx_int", "log_watch_ratio",
]].copy()
baseline_posterior_df["r_hat"] = final_r_hat.astype(np.float32)
baseline_posterior_df["tau_hat"] = final_tau_i.astype(np.float32)
baseline_posterior_df.to_parquet(OUTPUT_DIR / "structural_baseline_posteriors.parquet", index=False)

np.savez_compressed(
    OUTPUT_DIR / "structural_estimation_arrays.npz",
    sigmas_by_user=sigmas_by_user,
    session_beliefs=session_beliefs_matrix,
    obs_user_idx=obs_user_idx,
    obs_cat_idx=obs_cat_idx,
    obs_sess_idx=obs_sess_idx,
    obs_log_watch_ratio=obs_log_watch_ratio,
    user_ids=user_ids,
    cat_ids=cat_ids,
    reference_cat_idx=np.array([UTILITY_REFERENCE_CAT_IDX], dtype=np.int64),
)

np.savez_compressed(
    OUTPUT_DIR / "structural_estimates_theta_phi.npz",
    theta_b_cat=theta_corr["b_cat"],
    theta_U=theta_corr["U"],
    theta_V=theta_corr["V"],
    theta_c=theta_corr["c"],
    phi_alpha_mu0=phi_corr["alpha_mu0"],
    phi_alpha_mu1=phi_corr["alpha_mu1"],
    phi_beta_mu0=phi_corr["beta_mu0"],
    phi_beta_mu1=phi_corr["beta_mu1"],
    phi_alpha_logvar0=phi_corr["alpha_logvar0"],
    phi_alpha_logvar1=phi_corr["alpha_logvar1"],
    phi_beta_logvar0=phi_corr["beta_logvar0"],
    phi_beta_logvar1=phi_corr["beta_logvar1"],
    user_ids=user_ids,
    cat_ids=cat_ids,
    reference_cat_idx=np.array([UTILITY_REFERENCE_CAT_IDX], dtype=np.int64),
)

bundle_metadata = {
    "n_users": int(len(user_ids)),
    "n_categories": int(len(cat_ids)),
    "n_interactions": int(len(interaction_df)),
    "n_sessions": int(len(session_table)),
    "burn_in_days": int(BURN_IN_DAYS),
    "min_obs_per_user": int(MIN_OBS_PER_USER),
    "score_weight_spec": HEAD_WEIGHT_SPEC,
    "score_weight_source": SCORE_WEIGHT_SOURCE,
    "score_weights": {key: float(value) for key, value in SCORE_WEIGHTS.items()},
    "baseline_observed_ll": float(final_ll),
    "row_id_alignment": "row_id equals the original processed_data row index and gnn_data edge row",
    "paths": {
        "interactions": str(OUTPUT_DIR / "structural_interactions.parquet"),
        "sessions": str(OUTPUT_DIR / "structural_sessions.parquet"),
        "arrays": str(OUTPUT_DIR / "structural_estimation_arrays.npz"),
        "baseline_posteriors": str(OUTPUT_DIR / "structural_baseline_posteriors.parquet"),
    },
}
(OUTPUT_DIR / "structural_estimation_bundle_metadata.json").write_text(json.dumps(bundle_metadata, indent=2))

threshold_diag_df = pd.DataFrame(threshold_diagnostic_records)
threshold_diag_df.to_csv(OUTPUT_DIR / "threshold_residual_diagnostics.csv", index=False)

print("saved reusable structural bundle for notebook 07")
print(json.dumps(bundle_metadata, indent=2))


In [ ]:
# Explicit train/validation likelihood report.
# Run the session-level train/validation diagnostic cell above with RUN_SESSION_VALIDATION_DIAGNOSTIC = True first.

def print_train_validation_likelihood_report():
    required_names = ["train_ll_summary", "val_ll_summary"]
    missing = [name for name in required_names if name not in globals()]
    if missing:
        print("Train/validation likelihood report is not available yet.")
        print("Set RUN_SESSION_VALIDATION_DIAGNOSTIC = True in the session-level diagnostic cell above and run that cell first.")
        print(f"Missing objects: {missing}")
        return None

    report = pd.DataFrame([train_ll_summary, val_ll_summary])
    report_cols = [
        "split", "n_obs", "n_sessions", "total_ll", "avg_ll_per_obs",
        "avg_session_avg_ll", "median_session_avg_ll", "mean_r_hat",
        "corr_r_watch_ratio", "corr_r_log_watch_ratio",
    ]
    report = report[report_cols]

    gap_report = pd.DataFrame([{
        "avg_ll_per_obs_train": train_ll_summary["avg_ll_per_obs"],
        "avg_ll_per_obs_validation": val_ll_summary["avg_ll_per_obs"],
        "avg_ll_per_obs_gap_train_minus_validation": train_ll_summary["avg_ll_per_obs"] - val_ll_summary["avg_ll_per_obs"],
        "avg_session_ll_train": train_ll_summary["avg_session_avg_ll"],
        "avg_session_ll_validation": val_ll_summary["avg_session_avg_ll"],
        "avg_session_ll_gap_train_minus_validation": train_ll_summary["avg_session_avg_ll"] - val_ll_summary["avg_session_avg_ll"],
        "corr_r_watch_ratio_train": train_ll_summary["corr_r_watch_ratio"],
        "corr_r_watch_ratio_validation": val_ll_summary["corr_r_watch_ratio"],
    }])

    print("=== Train/Validation Observed Likelihood Diagnostic ===")
    print(f"Train observations: {train_ll_summary['n_obs']:,}; sessions: {train_ll_summary['n_sessions']:,}")
    print(f"Validation/test observations: {val_ll_summary['n_obs']:,}; sessions: {val_ll_summary['n_sessions']:,}")
    print(f"Train total observed LL: {train_ll_summary['total_ll']:.3f}")
    print(f"Validation/test total observed LL: {val_ll_summary['total_ll']:.3f}")
    print(f"Train avg LL per observation: {train_ll_summary['avg_ll_per_obs']:.6f}")
    print(f"Validation/test avg LL per observation: {val_ll_summary['avg_ll_per_obs']:.6f}")
    print(f"Gap, train minus validation/test: {gap_report.loc[0, 'avg_ll_per_obs_gap_train_minus_validation']:.6f}")
    print(f"corr(r_hat, watch_ratio), train: {train_ll_summary['corr_r_watch_ratio']:.6f}")
    print(f"corr(r_hat, watch_ratio), validation/test: {val_ll_summary['corr_r_watch_ratio']:.6f}")

    display(report)
    display(gap_report)
    return report, gap_report


_train_val_report_output = print_train_validation_likelihood_report()
if _train_val_report_output is not None:
    train_val_report, train_val_gap_report = _train_val_report_output


In [ ]:
# Diagnostics: orientation, cost distribution, and user 3586 mu-vs-score rank.

# 1. Orientation check: z=1 should be high-watch.
r_corr, tau_corr, ll_corr = compute_e_step_with_ll(
    interaction_df=interaction_df,
    session_table=session_table,
    theta=theta_corr,
    phi=phi_corr,
    sigmas_by_user=sigmas_by_user,
)

diag = interaction_df[["log_watch_ratio", "i_cat_level1_id"]].copy()
if "watch_ratio" in interaction_df.columns:
    diag["watch_ratio"] = pd.to_numeric(interaction_df["watch_ratio"], errors="coerce").to_numpy(dtype=np.float64)
else:
    diag["watch_ratio"] = np.exp(diag["log_watch_ratio"].to_numpy(dtype=np.float64))
diag["r_hat"] = r_corr

m0_corr, v0_corr, m1_corr, v1_corr = predict_watch_moments(interaction_df, phi_corr)

print("Final observed LL:", ll_corr)
print("corr(r_hat, watch_ratio):", diag["r_hat"].corr(diag["watch_ratio"]))
print("average predicted measurement mean z=0:", float(np.mean(m0_corr)))
print("average predicted measurement mean z=1:", float(np.mean(m1_corr)))
print("share m1 > m0:", float(np.mean(m1_corr > m0_corr)))

diag["r_decile"] = pd.qcut(diag["r_hat"], 10, duplicates="drop")
display(
    diag.groupby("r_decile", observed=True)["watch_ratio"]
    .agg(["count", "mean", "median"])
)

# r_hat distribution and binned watch-behavior diagnostic.
# Do not use a raw watch_ratio scatter plot: extreme ratios from short/replayed videos dominate the y-axis.
import matplotlib.pyplot as plt

plot_diag = diag.copy()
plot_diag["watch_ratio_capped"] = plot_diag["watch_ratio"].clip(upper=3.0)
plot_diag["r_bin"] = pd.qcut(plot_diag["r_hat"], 20, duplicates="drop")

r_bin_summary = (
    plot_diag
    .groupby("r_bin", observed=True)
    .agg(
        n_obs=("r_hat", "size"),
        r_mean=("r_hat", "mean"),
        r_min=("r_hat", "min"),
        r_max=("r_hat", "max"),
        watch_ratio_median=("watch_ratio", "median"),
        watch_ratio_capped_mean=("watch_ratio_capped", "mean"),
        log_watch_ratio_mean=("log_watch_ratio", "mean"),
        log_watch_ratio_median=("log_watch_ratio", "median"),
    )
    .reset_index()
)
r_bin_summary["geom_mean_watch_ratio"] = np.exp(r_bin_summary["log_watch_ratio_mean"])

display(r_bin_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(plot_diag["r_hat"], bins=100)
axes[0].set_title("Distribution of r_hat")
axes[0].set_xlabel("r_hat")
axes[0].set_ylabel("count")

axes[1].plot(r_bin_summary["r_mean"], r_bin_summary["watch_ratio_median"], marker="o", label="median watch ratio")
axes[1].plot(r_bin_summary["r_mean"], r_bin_summary["geom_mean_watch_ratio"], marker="o", label="exp(mean log watch ratio)")
axes[1].plot(r_bin_summary["r_mean"], r_bin_summary["watch_ratio_capped_mean"], marker="o", label="mean capped watch ratio")
axes[1].set_title("Binned watch behavior by r_hat")
axes[1].set_xlabel("mean r_hat in bin")
axes[1].set_ylabel("watch-ratio scale")
axes[1].legend()

plt.tight_layout()
plt.show()

# 2. Cost distribution.
cost_df = pd.DataFrame({
    "user_id": user_ids,
    "user_idx": np.arange(len(user_ids)),
    "c": theta_corr["c"],
    "log_c": np.log(np.maximum(theta_corr["c"], 1e-12)),
})

display(cost_df["c"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))
display(cost_df.sort_values("c").head(10))
display(cost_df.sort_values("c", ascending=False).head(10))

# 3. User 3586: compare structural mu rank vs normalized mean-score rank.
raw_user_id = 3586

if raw_user_id not in uid_to_idx:
    raise ValueError(f"user_id={raw_user_id} is not in the estimation sample")

user_idx = uid_to_idx[raw_user_id]
mu_matrix = theta_to_mu_matrix(theta_corr, n_users=N_users, k_cat=K_cat)

mu_rank_df = pd.DataFrame({
    "user_id": raw_user_id,
    "user_idx": user_idx,
    "i_cat_level1_id": cat_ids,
    "cat_idx": np.arange(K_cat),
    "mu_ik": mu_matrix[user_idx],
    "sigma_ik": sigmas_by_user[user_idx],
})

mu_rank_df = mu_rank_df.sort_values("mu_ik", ascending=False).reset_index(drop=True)
mu_rank_df["mu_rank"] = np.arange(1, len(mu_rank_df) + 1)

score_rank_df = (
    user_cat_score_moments
    .loc[user_cat_score_moments["user_id"].eq(raw_user_id)]
    [["user_id", "i_cat_level1_id", "mean_score", "var_score", "n_score"]]
    .copy()
)

score_rank_df = score_rank_df.sort_values("mean_score", ascending=False).reset_index(drop=True)
score_rank_df["mean_score_rank"] = np.arange(1, len(score_rank_df) + 1)

compare_rank_df = (
    mu_rank_df
    .merge(score_rank_df, on=["user_id", "i_cat_level1_id"], how="left")
)

compare_rank_df["mean_score_rank"] = compare_rank_df["mean_score_rank"].astype("Int64")
compare_rank_df["rank_gap_mu_minus_score"] = compare_rank_df["mu_rank"] - compare_rank_df["mean_score_rank"]

display(compare_rank_df[
    [
        "user_id",
        "user_idx",
        "i_cat_level1_id",
        "cat_idx",
        "mu_rank",
        "mu_ik",
        "mean_score_rank",
        "mean_score",
        "rank_gap_mu_minus_score",
        "n_score",
        "sigma_ik",
    ]
].sort_values("mu_rank"))

tmp = compare_rank_df.dropna(subset=["mean_score_rank"]).copy()
print("Spearman rank corr:", tmp["mu_rank"].corr(tmp["mean_score_rank"], method="spearman"))
print("Pearson corr mu vs mean_score:", tmp["mu_ik"].corr(tmp["mean_score"]))

print("\nCategory 8:")
display(compare_rank_df[compare_rank_df["i_cat_level1_id"].eq(8)])

print("\nCategory 24:")
display(compare_rank_df[compare_rank_df["i_cat_level1_id"].eq(24)])

print("\nUNKNOWN category:")
display(compare_rank_df[compare_rank_df["i_cat_level1_id"].eq(SCORE_REFERENCE_CATEGORY_ID)])

In [ ]:
# Inspect phi: measurement model coefficients and predicted z=0/z=1 behavior.

phi_hat = phi_corr

# Feature names
mean_names = phi_hat["feature_names_mean"]
logvar_names = phi_hat["feature_names_logvar"]

phi_mean_coef_df = pd.DataFrame({
    "feature": mean_names,
    "beta_mu0": phi_hat["beta_mu0"],
    "beta_mu1": phi_hat["beta_mu1"],
    "diff_mu1_minus_mu0": phi_hat["beta_mu1"] - phi_hat["beta_mu0"],
})

phi_logvar_coef_df = pd.DataFrame({
    "feature": logvar_names,
    "beta_logvar0": phi_hat["beta_logvar0"],
    "beta_logvar1": phi_hat["beta_logvar1"],
    "diff_logvar1_minus_logvar0": phi_hat["beta_logvar1"] - phi_hat["beta_logvar0"],
})

display(phi_mean_coef_df)
display(phi_logvar_coef_df)

# Category fixed effects
phi_cat_df = pd.DataFrame({
    "i_cat_level1_id": cat_ids,
    "cat_idx": np.arange(K_cat),
    "alpha_mu0": phi_hat["alpha_mu0"],
    "alpha_mu1": phi_hat["alpha_mu1"],
    "alpha_mu_diff_1_minus_0": phi_hat["alpha_mu1"] - phi_hat["alpha_mu0"],
    "alpha_logvar0": phi_hat["alpha_logvar0"],
    "alpha_logvar1": phi_hat["alpha_logvar1"],
    "alpha_logvar_diff_1_minus_0": phi_hat["alpha_logvar1"] - phi_hat["alpha_logvar0"],
})

display(phi_cat_df.sort_values("alpha_mu_diff_1_minus_0", ascending=False))